In [1]:
from pathlib import Path

import pandas as pd


# ---------------------------------------------------------------------------
# Project paths
# ---------------------------------------------------------------------------

PROJECT_ROOT = Path.cwd().resolve().parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

LOAN_FILE = RAW_DATA_DIR / "loan_portfolio.csv"


# ---------------------------------------------------------------------------
# Validate project structure
# ---------------------------------------------------------------------------

if not LOAN_FILE.exists():
    raise FileNotFoundError(
        f"Raw loan portfolio file not found: {LOAN_FILE}"
    )

PROCESSED_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data directory: {RAW_DATA_DIR}")
print(f"Processed data directory: {PROCESSED_DATA_DIR}")

Project root: D:\Bank_sense_2.0
Raw data directory: D:\Bank_sense_2.0\data\raw
Processed data directory: D:\Bank_sense_2.0\data\processed


In [2]:
# ---------------------------------------------------------------------------
# Load raw loan portfolio
# ---------------------------------------------------------------------------

loan_portfolio = pd.read_csv(
    LOAN_FILE,
    low_memory=False,
)

print(
    f"Loaded {len(loan_portfolio):,} loan records "
    f"with {len(loan_portfolio.columns)} columns."
)

Loaded 50,000 loan records with 24 columns.


In [3]:
# ---------------------------------------------------------------------------
# Create cleaning workspace
# ---------------------------------------------------------------------------

clean_loan_portfolio = loan_portfolio.copy()

print("Cleaning workspace created.")

Cleaning workspace created.


In [4]:
# ---------------------------------------------------------------------------
# Standardize column names
# ---------------------------------------------------------------------------

clean_loan_portfolio.columns = (
    clean_loan_portfolio.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^\w]+", "_", regex=True)
    .str.strip("_")
)

display(
    pd.DataFrame({
        "column": clean_loan_portfolio.columns
    })
)

,column
0,loan_id
1,origination_date
2,maturity_date
3,maturity_months
4,sector
5,loan_type
6,collateral
7,initial_rating
8,credit_score
9,ead


In [5]:
# ---------------------------------------------------------------------------
# Standardize date columns
# ---------------------------------------------------------------------------

DATE_COLUMNS = [
    "origination_date",
    "maturity_date",
    "default_date",
]

for column in DATE_COLUMNS:
    clean_loan_portfolio[column] = pd.to_datetime(
        clean_loan_portfolio[column],
        errors="coerce",
    )

print("Date columns converted successfully.")

Date columns converted successfully.


In [6]:
# ---------------------------------------------------------------------------
# Validate cleaned data types
# ---------------------------------------------------------------------------

date_type_check = clean_loan_portfolio[DATE_COLUMNS].dtypes

display(
    date_type_check.to_frame("dtype")
)

,dtype
origination_date,datetime64[us]
maturity_date,datetime64[us]
default_date,datetime64[us]


In [7]:
# ---------------------------------------------------------------------------
# Standardize categorical columns
# ---------------------------------------------------------------------------

CATEGORICAL_COLUMNS = [
    "sector",
    "loan_type",
    "collateral",
    "initial_rating",
]

for column in CATEGORICAL_COLUMNS:
    clean_loan_portfolio[column] = (
        clean_loan_portfolio[column]
        .astype("string")
        .str.strip()
    )

print("Categorical columns standardized.")

Categorical columns standardized.


In [8]:
# ---------------------------------------------------------------------------
# Validate categorical values
# ---------------------------------------------------------------------------

for column in CATEGORICAL_COLUMNS:
    print(f"\n{column}")
    display(
        clean_loan_portfolio[column]
        .value_counts(dropna=False)
        .rename_axis(column)
        .reset_index(name="count")
    )


sector


,sector,count
0,Energy,5132
1,Telecom,5077
2,Consumer,5043
3,Financials,5023
4,Industrials,5012
5,Healthcare,4982
6,Technology,4959
7,Utilities,4959
8,Retail,4951
9,Real_Estate,4862



loan_type


,loan_type,count
0,term_loan,17594
1,mortgage,12262
2,revolving,10158
3,bond,7458
4,lease,2528



collateral


,collateral,count
0,secured,22657
1,unsecured,17398
2,partially_secured,9945



initial_rating


,initial_rating,count
0,BBB,14001
1,BB,10918
2,A,7571
3,B,7559
4,CCC,4478
5,AA,3944
6,AAA,1529


In [9]:
# ---------------------------------------------------------------------------
# Create data-quality flags
# ---------------------------------------------------------------------------

clean_loan_portfolio["post_maturity_default"] = (
    clean_loan_portfolio["default_date"]
    > clean_loan_portfolio["maturity_date"]
)

clean_loan_portfolio["default_date_missing_for_default"] = (
    clean_loan_portfolio["defaulted"].eq(1)
    & clean_loan_portfolio["default_date"].isna()
)

clean_loan_portfolio["default_date_present_for_non_default"] = (
    clean_loan_portfolio["defaulted"].eq(0)
    & clean_loan_portfolio["default_date"].notna()
)

clean_loan_portfolio["invalid_maturity_sequence"] = (
    clean_loan_portfolio["maturity_date"]
    < clean_loan_portfolio["origination_date"]
)

clean_loan_portfolio["invalid_default_sequence"] = (
    clean_loan_portfolio["default_date"]
    < clean_loan_portfolio["origination_date"]
)

quality_flags = [
    "post_maturity_default",
    "default_date_missing_for_default",
    "default_date_present_for_non_default",
    "invalid_maturity_sequence",
    "invalid_default_sequence",
]

display(
    clean_loan_portfolio[quality_flags]
    .sum()
    .rename("flagged_records")
    .to_frame()
)

,flagged_records
post_maturity_default,662
default_date_missing_for_default,0
default_date_present_for_non_default,0
invalid_maturity_sequence,0
invalid_default_sequence,0


In [10]:
# ---------------------------------------------------------------------------
# Standardize numeric columns
# ---------------------------------------------------------------------------

NUMERIC_COLUMNS = [
    "maturity_months",
    "credit_score",
    "ead",
    "coupon_rate",
    "leverage",
    "interest_coverage",
    "debt_to_equity",
    "pd_annual",
    "lgd",
    "el",
    "unexpected_loss",
    "rwa",
    "survival_months",
    "recovery_rate",
    "loss_given_default",
]

for column in NUMERIC_COLUMNS:
    clean_loan_portfolio[column] = pd.to_numeric(
        clean_loan_portfolio[column],
        errors="coerce",
    )

print("Numeric columns standardized.")

Numeric columns standardized.


In [11]:
# ---------------------------------------------------------------------------
# Validate numeric data types
# ---------------------------------------------------------------------------

numeric_type_check = (
    clean_loan_portfolio[NUMERIC_COLUMNS]
    .dtypes
    .astype(str)
    .rename("dtype")
    .to_frame()
)

display(numeric_type_check)

,dtype
maturity_months,int64
credit_score,int64
ead,float64
coupon_rate,float64
leverage,float64
interest_coverage,float64
debt_to_equity,float64
pd_annual,float64
lgd,float64
el,float64


In [12]:
# ---------------------------------------------------------------------------
# Post-conversion missing-value check
# ---------------------------------------------------------------------------

conversion_missing = (
    clean_loan_portfolio[NUMERIC_COLUMNS]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

display(
    conversion_missing
    .rename("missing_count")
    .to_frame()
)

,missing_count
recovery_rate,43050
loss_given_default,43050
maturity_months,0
coupon_rate,0
leverage,0
credit_score,0
ead,0
debt_to_equity,0
interest_coverage,0
pd_annual,0


In [13]:
# ---------------------------------------------------------------------------
# Compare missing values before and after type standardization
# ---------------------------------------------------------------------------

missing_comparison = pd.DataFrame(
    {
        "raw_missing": loan_portfolio[NUMERIC_COLUMNS].isna().sum(),
        "clean_missing": clean_loan_portfolio[NUMERIC_COLUMNS].isna().sum(),
    }
)

missing_comparison["change"] = (
    missing_comparison["clean_missing"]
    - missing_comparison["raw_missing"]
)

display(missing_comparison)

,raw_missing,clean_missing,change
maturity_months,0,0,0
credit_score,0,0,0
ead,0,0,0
coupon_rate,0,0,0
leverage,0,0,0
interest_coverage,0,0,0
debt_to_equity,0,0,0
pd_annual,0,0,0
lgd,0,0,0
el,0,0,0


### Missing Numeric Values — Cleaning Decision

Numeric type standardization did not introduce any new missing values.

The only missing numeric fields are:

- `recovery_rate`: 43,050 records
- `loss_given_default`: 43,050 records

These missing values correspond exactly to the 43,050 non-defaulted loans.

Because recovery and realized loss are post-default concepts, the missing values
are considered **structurally valid** rather than data-quality errors.

**Decision:** Do not impute these values. Preserve them as missing (`NaN`) and
use default-status logic when analysing or modelling recovery/LGD.
    

In [15]:
# ---------------------------------------------------------------------------
# Standardize target and boolean fields
# ---------------------------------------------------------------------------

clean_loan_portfolio["defaulted"] = (
    clean_loan_portfolio["defaulted"]
    .astype("int8")
)

clean_loan_portfolio["post_maturity_default"] = (
    clean_loan_portfolio["post_maturity_default"]
    .astype("boolean")
)

clean_loan_portfolio["default_date_missing_for_default"] = (
    clean_loan_portfolio["default_date_missing_for_default"]
    .astype("boolean")
)

clean_loan_portfolio["default_date_present_for_non_default"] = (
    clean_loan_portfolio["default_date_present_for_non_default"]
    .astype("boolean")
)

clean_loan_portfolio["invalid_maturity_sequence"] = (
    clean_loan_portfolio["invalid_maturity_sequence"]
    .astype("boolean")
)

clean_loan_portfolio["invalid_default_sequence"] = (
    clean_loan_portfolio["invalid_default_sequence"]
    .astype("boolean")
)

print("Target and quality-flag dtypes standardized.")

Target and quality-flag dtypes standardized.


In [16]:
# ---------------------------------------------------------------------------
# Validate target and quality-flag values
# ---------------------------------------------------------------------------

flag_columns = [
    "defaulted",
    "post_maturity_default",
    "default_date_missing_for_default",
    "default_date_present_for_non_default",
    "invalid_maturity_sequence",
    "invalid_default_sequence",
]

display(
    clean_loan_portfolio[flag_columns]
    .dtypes
    .astype(str)
    .rename("dtype")
    .to_frame()
)

,dtype
defaulted,int8
post_maturity_default,boolean
default_date_missing_for_default,boolean
default_date_present_for_non_default,boolean
invalid_maturity_sequence,boolean
invalid_default_sequence,boolean


In [17]:
# ---------------------------------------------------------------------------
# Validate numeric ranges after cleaning
# ---------------------------------------------------------------------------

range_checks = pd.DataFrame(
    {
        "check": [
            "credit_score < 0",
            "credit_score > 1000",
            "maturity_months <= 0",
            "ead <= 0",
            "coupon_rate < 0",
            "leverage < 0",
            "interest_coverage < 0",
            "debt_to_equity < 0",
            "pd_annual < 0",
            "pd_annual > 1",
            "lgd < 0",
            "lgd > 1",
            "recovery_rate < 0",
            "recovery_rate > 1",
            "defaulted not in {0, 1}",
        ],
        "count": [
            clean_loan_portfolio["credit_score"].lt(0).sum(),
            clean_loan_portfolio["credit_score"].gt(1000).sum(),
            clean_loan_portfolio["maturity_months"].le(0).sum(),
            clean_loan_portfolio["ead"].le(0).sum(),
            clean_loan_portfolio["coupon_rate"].lt(0).sum(),
            clean_loan_portfolio["leverage"].lt(0).sum(),
            clean_loan_portfolio["interest_coverage"].lt(0).sum(),
            clean_loan_portfolio["debt_to_equity"].lt(0).sum(),
            clean_loan_portfolio["pd_annual"].lt(0).sum(),
            clean_loan_portfolio["pd_annual"].gt(1).sum(),
            clean_loan_portfolio["lgd"].lt(0).sum(),
            clean_loan_portfolio["lgd"].gt(1).sum(),
            clean_loan_portfolio["recovery_rate"].lt(0).sum(),
            clean_loan_portfolio["recovery_rate"].gt(1).sum(),
            (~clean_loan_portfolio["defaulted"].isin([0, 1])).sum(),
        ],
    }
)

display(range_checks)

,check,count
0,credit_score < 0,0
1,credit_score > 1000,0
2,maturity_months <= 0,0
3,ead <= 0,0
4,coupon_rate < 0,0
5,leverage < 0,0
6,interest_coverage < 0,0
7,debt_to_equity < 0,0
8,pd_annual < 0,0
9,pd_annual > 1,0


### Numeric Range Validation — Finding

All defined numeric range checks returned **zero violations** after
standardization.

The cleaned dataset contains no invalid values for:

- credit score bounds
- maturity duration
- EAD positivity
- coupon rate
- leverage
- interest coverage
- debt-to-equity
- PD bounds
- LGD bounds
- recovery-rate bounds
- binary default status

**Conclusion:** No numeric range corrections are required for the audited fields.


In [20]:
# ---------------------------------------------------------------------------
# Validate core financial relationships
# ---------------------------------------------------------------------------

financial_checks = pd.Series(
    {
        "lgd_recovery_mismatch": int(
            (
                clean_loan_portfolio.loc[
                    clean_loan_portfolio["defaulted"].eq(1),
                    "lgd",
                ]
                - (
                    1
                    - clean_loan_portfolio.loc[
                        clean_loan_portfolio["defaulted"].eq(1),
                        "recovery_rate",
                    ]
                )
            )
            .abs()
            .gt(1e-10)
            .sum()
        ),
        "loss_lgd_mismatch": int(
            (
                clean_loan_portfolio.loc[
                    clean_loan_portfolio["defaulted"].eq(1),
                    "loss_given_default",
                ]
                - (
                    clean_loan_portfolio.loc[
                        clean_loan_portfolio["defaulted"].eq(1),
                        "ead",
                    ]
                    * clean_loan_portfolio.loc[
                        clean_loan_portfolio["defaulted"].eq(1),
                        "lgd",
                    ]
                )
            )
            .abs()
            .gt(1e-2)
            .sum()
        ),
    },
    name="count",
)

display(financial_checks.to_frame())

,count
lgd_recovery_mismatch,0
loss_lgd_mismatch,0


### Financial Relationship Validation — Finding

The cleaned loan portfolio preserves the core financial relationships identified
during the data audit.

- LGD / recovery mismatches: **0**
- Loss / LGD mismatches: **0**

**Conclusion:** Cleaning did not introduce inconsistencies into the core
credit-risk calculations.

In [19]:
# ---------------------------------------------------------------------------
# Validate temporal quality flags
# ---------------------------------------------------------------------------

temporal_flag_summary = (
    clean_loan_portfolio[
        [
            "post_maturity_default",
            "default_date_missing_for_default",
            "default_date_present_for_non_default",
            "invalid_maturity_sequence",
            "invalid_default_sequence",
        ]
    ]
    .sum()
    .rename("flagged_records")
    .to_frame()
)

display(temporal_flag_summary)

,flagged_records
post_maturity_default,662
default_date_missing_for_default,0
default_date_present_for_non_default,0
invalid_maturity_sequence,0
invalid_default_sequence,0



### Temporal Validation — Finding

The cleaned dataset preserves all valid temporal relationships identified
during the audit.

- Defaulted loans missing `default_date`: **0**
- Non-defaulted loans with `default_date`: **0**
- Maturity before origination: **0**
- Default before origination: **0**
- Post-maturity defaults: **662**

The 662 post-maturity defaults are retained and explicitly flagged because
they represent a structural feature of the source dataset rather than a
confirmed data-entry error.

**Conclusion:** The cleaning process did not alter valid dates or remove the
identified temporal anomaly.

In [21]:
# ---------------------------------------------------------------------------
# Save cleaned loan portfolio
# ---------------------------------------------------------------------------

CLEAN_LOAN_FILE = (
    PROCESSED_DATA_DIR
    / "loan_portfolio_clean.csv"
)

clean_loan_portfolio.to_csv(
    CLEAN_LOAN_FILE,
    index=False,
)

print(
    f"Cleaned loan portfolio saved to:\n{CLEAN_LOAN_FILE}"
)

Cleaned loan portfolio saved to:
D:\Bank_sense_2.0\data\processed\loan_portfolio_clean.csv


In [22]:
# ---------------------------------------------------------------------------
# Verify processed output
# ---------------------------------------------------------------------------

if not CLEAN_LOAN_FILE.exists():
    raise FileNotFoundError(
        f"Processed file was not created: {CLEAN_LOAN_FILE}"
    )

print(
    f"Verified: {CLEAN_LOAN_FILE.name}"
)
print(
    f"File size: "
    f"{CLEAN_LOAN_FILE.stat().st_size / (1024 ** 2):.2f} MB"
)

Verified: loan_portfolio_clean.csv
File size: 8.83 MB


In [23]:
# ---------------------------------------------------------------------------
# Load raw credit-ratings data
# ---------------------------------------------------------------------------

CREDIT_RATINGS_FILE = (
    RAW_DATA_DIR / "credit_ratings.csv"
)

if not CREDIT_RATINGS_FILE.exists():
    raise FileNotFoundError(
        f"Credit ratings file not found: {CREDIT_RATINGS_FILE}"
    )

credit_ratings = pd.read_csv(
    CREDIT_RATINGS_FILE,
    low_memory=False,
)

print(
    f"Loaded {len(credit_ratings):,} credit-rating records "
    f"with {len(credit_ratings.columns)} columns."
)

Loaded 17,939 credit-rating records with 9 columns.


In [24]:
# ---------------------------------------------------------------------------
# Create cleaning workspace
# ---------------------------------------------------------------------------

clean_credit_ratings = credit_ratings.copy()

print("Credit-ratings cleaning workspace created.")

Credit-ratings cleaning workspace created.


In [25]:
# ---------------------------------------------------------------------------
# Create cleaning workspace
# ---------------------------------------------------------------------------

clean_credit_ratings = credit_ratings.copy()

print("Credit-ratings cleaning workspace created.")

Credit-ratings cleaning workspace created.


In [26]:
# ---------------------------------------------------------------------------
# Initial credit-ratings inspection
# ---------------------------------------------------------------------------

credit_ratings_profile = pd.DataFrame(
    {
        "column": clean_credit_ratings.columns,
        "dtype": clean_credit_ratings.dtypes.astype(str).values,
        "missing_count": clean_credit_ratings.isna().sum().values,
        "unique_values": clean_credit_ratings.nunique(
            dropna=False
        ).values,
    }
)

display(credit_ratings_profile)

,column,dtype,missing_count,unique_values
0,issuer_id,str,0,2000
1,sector,str,0,10
2,year,int64,0,10
3,from_rating,str,0,7
4,to_rating,str,0,8
5,upgraded,int64,0,2
6,downgraded,int64,0,2
7,defaulted,int64,0,2
8,notches_moved,int64,0,13


In [27]:
# ---------------------------------------------------------------------------
# Standardize credit-ratings column names
# ---------------------------------------------------------------------------

clean_credit_ratings.columns = (
    clean_credit_ratings.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^\w]+", "_", regex=True)
    .str.strip("_")
)

display(
    pd.DataFrame(
        {"column": clean_credit_ratings.columns}
    )
)

,column
0,issuer_id
1,sector
2,year
3,from_rating
4,to_rating
5,upgraded
6,downgraded
7,defaulted
8,notches_moved


In [28]:
# ---------------------------------------------------------------------------
# Duplicate-row check
# ---------------------------------------------------------------------------

duplicate_count = clean_credit_ratings.duplicated().sum()

print(
    f"Duplicate credit-rating rows: {duplicate_count:,}"
)

Duplicate credit-rating rows: 0


In [29]:
# ---------------------------------------------------------------------------
# Inspect credit-rating columns
# ---------------------------------------------------------------------------

for column in clean_credit_ratings.columns:
    print(
        f"{column}: "
        f"{clean_credit_ratings[column].nunique(dropna=False):,} "
        f"unique values"
    )

issuer_id: 2,000 unique values
sector: 10 unique values
year: 10 unique values
from_rating: 7 unique values
to_rating: 8 unique values
upgraded: 2 unique values
downgraded: 2 unique values
defaulted: 2 unique values
notches_moved: 13 unique values


In [30]:
# ---------------------------------------------------------------------------
# Rating-value inspection
# ---------------------------------------------------------------------------

RATING_COLUMNS = [
    "from_rating",
    "to_rating",
]

for column in RATING_COLUMNS:
    if column in clean_credit_ratings.columns:
        print(f"\n{column}")
        display(
            clean_credit_ratings[column]
            .value_counts(dropna=False)
            .rename_axis(column)
            .reset_index(name="count")
        )


from_rating


,from_rating,count
0,BBB,4735
1,A,3794
2,BB,3233
3,B,3202
4,AA,1657
5,CCC,862
6,AAA,456



to_rating


,to_rating,count
0,BBB,4587
1,A,3914
2,B,3178
3,BB,3044
4,AA,1640
5,CCC,729
6,AAA,430
7,D,417


In [31]:
# ---------------------------------------------------------------------------
# Standardize credit-rating categorical fields
# ---------------------------------------------------------------------------

CREDIT_RATING_CATEGORICAL_COLUMNS = [
    "sector",
    "from_rating",
    "to_rating",
]

for column in CREDIT_RATING_CATEGORICAL_COLUMNS:
    clean_credit_ratings[column] = (
        clean_credit_ratings[column]
        .astype("string")
        .str.strip()
    )

print("Credit-rating categorical fields standardized.")

Credit-rating categorical fields standardized.


In [32]:
# ---------------------------------------------------------------------------
# Standardize numeric fields
# ---------------------------------------------------------------------------

CREDIT_RATING_NUMERIC_COLUMNS = [
    "year",
    "upgraded",
    "downgraded",
    "defaulted",
    "notches_moved",
]

for column in CREDIT_RATING_NUMERIC_COLUMNS:
    clean_credit_ratings[column] = pd.to_numeric(
        clean_credit_ratings[column],
        errors="coerce",
    )

print("Credit-rating numeric fields standardized.")

Credit-rating numeric fields standardized.


In [33]:
# ---------------------------------------------------------------------------
# Validate binary flags
# ---------------------------------------------------------------------------

flag_columns = [
    "upgraded",
    "downgraded",
    "defaulted",
]

flag_validation = pd.DataFrame(
    {
        column: {
            "missing": clean_credit_ratings[column].isna().sum(),
            "invalid_values": (
                ~clean_credit_ratings[column].isin([0, 1])
            ).sum(),
        }
        for column in flag_columns
    }
).T

display(flag_validation)

,missing,invalid_values
upgraded,0,0
downgraded,0,0
defaulted,0,0


In [34]:
# ---------------------------------------------------------------------------
# Define rating hierarchy
# ---------------------------------------------------------------------------

RATING_ORDER = {
    "AAA": 7,
    "AA": 6,
    "A": 5,
    "BBB": 4,
    "BB": 3,
    "B": 2,
    "CCC": 1,
    "D": 0,
}

clean_credit_ratings["from_score"] = (
    clean_credit_ratings["from_rating"].map(RATING_ORDER)
)

clean_credit_ratings["to_score"] = (
    clean_credit_ratings["to_rating"].map(RATING_ORDER)
)

clean_credit_ratings["calculated_notches_moved"] = (
    clean_credit_ratings["to_score"]
    - clean_credit_ratings["from_score"]
)

display(
    clean_credit_ratings[
        [
            "from_rating",
            "to_rating",
            "notches_moved",
            "calculated_notches_moved",
        ]
    ].head(20)
)

,from_rating,to_rating,notches_moved,calculated_notches_moved
0,A,A,0,0
1,A,A,0,0
2,BBB,BB,1,-1
3,BBB,BBB,0,0
4,A,A,0,0
5,B,B,0,0
6,BB,BB,0,0
7,BB,BBB,-1,1
8,BBB,BBB,0,0
9,AAA,AAA,0,0


In [35]:
# ---------------------------------------------------------------------------
# Validate notch movement
# ---------------------------------------------------------------------------

notch_difference = (
    clean_credit_ratings["notches_moved"]
    - clean_credit_ratings["calculated_notches_moved"]
)

print(
    f"Notch-movement mismatches: "
    f"{notch_difference.ne(0).sum():,}"
)

display(
    notch_difference.value_counts()
    .sort_index()
    .rename_axis("difference")
    .reset_index(name="count")
)

Notch-movement mismatches: 2,770


,difference,count
0,-12,1
1,-8,2
2,-6,21
3,-4,71
4,-2,1018
5,0,15169
6,2,1225
7,4,319
8,6,76
9,8,19


In [36]:
# ---------------------------------------------------------------------------
# Validate rating-migration flags
# ---------------------------------------------------------------------------

migration_checks = pd.Series(
    {
        "upgrade_flag_mismatch": (
            clean_credit_ratings["upgraded"]
            != clean_credit_ratings["calculated_notches_moved"].gt(0).astype(int)
        ).sum(),

        "downgrade_flag_mismatch": (
            clean_credit_ratings["downgraded"]
            != clean_credit_ratings["calculated_notches_moved"].lt(0).astype(int)
        ).sum(),

        "default_flag_vs_D_mismatch": (
            clean_credit_ratings["defaulted"]
            != clean_credit_ratings["to_rating"].eq("D").astype(int)
        ).sum(),
    },
    name="count",
)

display(migration_checks.to_frame())

,count
upgrade_flag_mismatch,0
downgrade_flag_mismatch,417
default_flag_vs_D_mismatch,0


In [37]:
# ---------------------------------------------------------------------------
# Investigate downgrade-flag mismatches
# ---------------------------------------------------------------------------

downgrade_mismatches = clean_credit_ratings.loc[
    (
        clean_credit_ratings["downgraded"]
        != clean_credit_ratings[
            "calculated_notches_moved"
        ].lt(0).astype(int)
    ),
    [
        "issuer_id",
        "sector",
        "year",
        "from_rating",
        "to_rating",
        "upgraded",
        "downgraded",
        "defaulted",
        "notches_moved",
        "calculated_notches_moved",
    ],
].copy()

print(
    f"Downgrade-flag mismatches: "
    f"{len(downgrade_mismatches):,}"
)

display(
    downgrade_mismatches.head(20)
)

Downgrade-flag mismatches: 417


,issuer_id,sector,year,from_rating,to_rating,upgraded,downgraded,defaulted,notches_moved,calculated_notches_moved
22,I00023,Financials,2015,CCC,D,0,0,1,1,-1
23,I00024,Financials,2015,B,D,0,0,1,2,-2
25,I00026,Telecom,2015,B,D,0,0,1,2,-2
34,I00035,Industrials,2015,CCC,D,0,0,1,1,-1
86,I00087,Technology,2015,CCC,D,0,0,1,1,-1
88,I00089,Retail,2015,B,D,0,0,1,2,-2
162,I00163,Technology,2015,CCC,D,0,0,1,1,-1
179,I00180,Energy,2015,B,D,0,0,1,2,-2
221,I00222,Financials,2015,CCC,D,0,0,1,1,-1
222,I00223,Financials,2015,CCC,D,0,0,1,1,-1


In [38]:
# ---------------------------------------------------------------------------
# Inspect mismatch transitions
# ---------------------------------------------------------------------------

mismatch_transition_counts = (
    downgrade_mismatches
    .groupby(
        ["from_rating", "to_rating"]
    )
    .size()
    .sort_values(ascending=False)
    .rename("count")
    .reset_index()
)

display(mismatch_transition_counts)

,from_rating,to_rating,count
0,CCC,D,179
1,B,D,172
2,BB,D,38
3,BBB,D,13
4,A,D,12
5,AA,D,2
6,AAA,D,1


### Credit-Rating Migration Validation — Finding

The 417 apparent downgrade-flag mismatches are entirely explained by transitions
to the default rating `D`.

All 417 observations have:

- `to_rating = D`
- `defaulted = 1`
- `upgraded = 0`
- `downgraded = 0`

The source dataset therefore treats **default migration as a separate event from
an ordinary credit downgrade**.

**Decision:** Do not modify the supplied `downgraded` flag.

The source `notches_moved` field also uses a dataset-specific encoding and does
not represent a conventional signed rating change for all transitions. The
source field will be preserved unchanged. A separate standardized
`rating_movement` variable can be derived later for analytical purposes.

In [39]:
# ---------------------------------------------------------------------------
# Validate issuer-year grain
# ---------------------------------------------------------------------------

issuer_year_duplicates = (
    clean_credit_ratings
    .groupby(["issuer_id", "year"])
    .size()
    .reset_index(name="record_count")
)

duplicate_issuer_years = issuer_year_duplicates.loc[
    issuer_year_duplicates["record_count"] > 1
]

print(
    f"Issuer-year combinations with multiple records: "
    f"{len(duplicate_issuer_years):,}"
)

display(
    duplicate_issuer_years.head(20)
)

Issuer-year combinations with multiple records: 0


,issuer_id,year,record_count


### Credit-Ratings Grain and Key Validation

No issuer-year combination appears more than once.

Therefore, the analytical grain of `credit_ratings` is:

**1 row = 1 issuer rating transition for 1 year**

The composite key is:

**(`issuer_id`, `year`)**

`issuer_id` is therefore not expected to be unique by itself.

**Decision:** Preserve all records. No duplicate removal is required for the issuer-year grain.

In [40]:
# ---------------------------------------------------------------------------
# Validate credit-rating year range
# ---------------------------------------------------------------------------

year_summary = pd.Series(
    {
        "minimum_year": clean_credit_ratings["year"].min(),
        "maximum_year": clean_credit_ratings["year"].max(),
        "unique_years": clean_credit_ratings["year"].nunique(),
        "missing_years": clean_credit_ratings["year"].isna().sum(),
    },
    name="value",
)

display(year_summary.to_frame())

,value
minimum_year,2015
maximum_year,2024
unique_years,10
missing_years,0


In [41]:
# ---------------------------------------------------------------------------
# Validate annual issuer coverage
# ---------------------------------------------------------------------------

issuer_coverage = (
    clean_credit_ratings
    .groupby("issuer_id")["year"]
    .agg(
        first_year="min",
        last_year="max",
        observations="count",
    )
)

display(issuer_coverage.describe())

,first_year,last_year,observations
count,2000.0,2000.000000,2000.000000
mean,2015.0,2022.969500,8.969500
std,0.0,2.412567,2.412567
min,2015.0,2015.000000,1.000000
25%,2015.0,2024.000000,10.000000
50%,2015.0,2024.000000,10.000000
75%,2015.0,2024.000000,10.000000
max,2015.0,2024.000000,10.000000


### Credit-Ratings Coverage — Finding

The `credit_ratings` dataset covers **2,000 issuers** across the period
**2015–2024**.

All issuers have observations beginning in 2015, but the number of annual
observations varies from **1 to 10**, making this an **unbalanced panel**.

**Decision:** Do not impute missing issuer-years or assume that an unobserved
year represents a stable rating. Rating-migration analysis will use observed
issuer-year transitions only.

In [42]:
# ---------------------------------------------------------------------------
# Validate year-to-year continuity for rating transitions
# ---------------------------------------------------------------------------

rating_sequence = (
    clean_credit_ratings
    .sort_values(["issuer_id", "year"])
    .copy()
)

rating_sequence["previous_year"] = (
    rating_sequence
    .groupby("issuer_id")["year"]
    .shift(1)
)

rating_sequence["year_gap"] = (
    rating_sequence["year"]
    - rating_sequence["previous_year"]
)

year_gap_summary = (
    rating_sequence.loc[
        rating_sequence["previous_year"].notna(),
        "year_gap",
    ]
    .value_counts()
    .sort_index()
)

display(
    year_gap_summary.rename("count").to_frame()
)

,count
year_gap,
1.0,15939


In [43]:
# ---------------------------------------------------------------------------
# Identify non-consecutive rating observations
# ---------------------------------------------------------------------------

non_consecutive_ratings = rating_sequence.loc[
    rating_sequence["year_gap"].gt(1),
    [
        "issuer_id",
        "sector",
        "year",
        "previous_year",
        "year_gap",
        "from_rating",
        "to_rating",
    ],
].copy()

print(
    f"Non-consecutive issuer-year observations: "
    f"{len(non_consecutive_ratings):,}"
)

display(
    non_consecutive_ratings.head(20)
)

Non-consecutive issuer-year observations: 0


,issuer_id,sector,year,previous_year,year_gap,from_rating,to_rating


### Credit-Rating Transition Continuity — Finding

All 15,939 observed issuer-to-next-year rating transitions occur over exactly
one year.

- One-year gaps: **15,939**
- Non-consecutive gaps: **0**

**Conclusion:** The credit-rating dataset provides a continuous annual
transition structure for all observed issuer-year sequences. No interpolation
or special handling of year gaps is required for the observed transitions.

In [44]:
# ---------------------------------------------------------------------------
# Create rating-transition classification
# ---------------------------------------------------------------------------

clean_credit_ratings["rating_transition_type"] = "unchanged"

clean_credit_ratings.loc[
    clean_credit_ratings["upgraded"].eq(1),
    "rating_transition_type",
] = "upgrade"

clean_credit_ratings.loc[
    clean_credit_ratings["downgraded"].eq(1),
    "rating_transition_type",
] = "downgrade"

clean_credit_ratings.loc[
    clean_credit_ratings["defaulted"].eq(1),
    "rating_transition_type",
] = "default"

In [45]:
# ---------------------------------------------------------------------------
# Save cleaned credit-ratings data
# ---------------------------------------------------------------------------

CLEAN_CREDIT_RATINGS_FILE = (
    PROCESSED_DATA_DIR / "credit_ratings_clean.csv"
)

clean_credit_ratings.to_csv(
    CLEAN_CREDIT_RATINGS_FILE,
    index=False,
)

print(
    f"Saved cleaned credit ratings to:\n"
    f"{CLEAN_CREDIT_RATINGS_FILE}"
)

Saved cleaned credit ratings to:
D:\Bank_sense_2.0\data\processed\credit_ratings_clean.csv


In [46]:
# ---------------------------------------------------------------------------
# Save cleaned credit-ratings data
# ---------------------------------------------------------------------------

CLEAN_CREDIT_RATINGS_FILE = (
    PROCESSED_DATA_DIR / "credit_ratings_clean.csv"
)

clean_credit_ratings.to_csv(
    CLEAN_CREDIT_RATINGS_FILE,
    index=False,
)

print(
    f"Saved cleaned credit ratings to:\n"
    f"{CLEAN_CREDIT_RATINGS_FILE}"
)

Saved cleaned credit ratings to:
D:\Bank_sense_2.0\data\processed\credit_ratings_clean.csv


In [47]:
# ---------------------------------------------------------------------------
# Verify processed output
# ---------------------------------------------------------------------------

if not CLEAN_CREDIT_RATINGS_FILE.exists():
    raise FileNotFoundError(
        f"Processed file was not created: "
        f"{CLEAN_CREDIT_RATINGS_FILE}"
    )

print(
    f"Verified: {CLEAN_CREDIT_RATINGS_FILE.name}"
)

Verified: credit_ratings_clean.csv


In [48]:
# ---------------------------------------------------------------------------
# Load raw macro stress scenarios
# ---------------------------------------------------------------------------

MACRO_STRESS_FILE = (
    RAW_DATA_DIR / "macro_stress_scenarios.csv"
)

if not MACRO_STRESS_FILE.exists():
    raise FileNotFoundError(
        f"Macro stress scenario file not found: {MACRO_STRESS_FILE}"
    )

macro_stress_scenarios = pd.read_csv(
    MACRO_STRESS_FILE,
    low_memory=False,
)

print(
    f"Loaded {len(macro_stress_scenarios):,} macro stress records "
    f"with {len(macro_stress_scenarios.columns)} columns."
)

Loaded 60 macro stress records with 16 columns.


In [49]:
# ---------------------------------------------------------------------------
# Create cleaning workspace
# ---------------------------------------------------------------------------

clean_macro_stress = macro_stress_scenarios.copy()

print("Macro stress cleaning workspace created.")

Macro stress cleaning workspace created.


In [50]:
# ---------------------------------------------------------------------------
# Initial macro stress data profile
# ---------------------------------------------------------------------------

macro_profile = pd.DataFrame(
    {
        "column": clean_macro_stress.columns,
        "dtype": clean_macro_stress.dtypes.astype(str).values,
        "missing_count": clean_macro_stress.isna().sum().values,
        "unique_values": clean_macro_stress.nunique(
            dropna=False
        ).values,
    }
)

display(macro_profile)

,column,dtype,missing_count,unique_values
0,scenario,str,0,6
1,gdp_shock_pp,float64,0,6
2,unemp_shock_pp,float64,0,6
3,rate_shock_pp,float64,0,6
4,credit_spread_bps,int64,0,6
5,sector,str,0,10
6,base_pd,float64,0,10
7,stressed_pd,float64,0,60
8,pd_uplift_pp,float64,0,51
9,pd_multiplier,float64,0,45


In [51]:
# ---------------------------------------------------------------------------
# Inspect macro stress categories
# ---------------------------------------------------------------------------

for column in clean_macro_stress.select_dtypes(
    include=["object", "string"]
).columns:
    print(f"\n{column}")

    display(
        clean_macro_stress[column]
        .value_counts(dropna=False)
        .rename_axis(column)
        .reset_index(name="count")
    )


scenario


,scenario,count
0,baseline,10
1,mild,10
2,adverse,10
3,severe,10
4,gfc_like,10
5,covid_like,10



sector


,sector,count
0,Financials,6
1,Real_Estate,6
2,Consumer,6
3,Industrials,6
4,Technology,6
5,Energy,6
6,Healthcare,6
7,Utilities,6
8,Retail,6
9,Telecom,6


In [52]:
# ---------------------------------------------------------------------------
# Standardize macro stress categorical fields
# ---------------------------------------------------------------------------

MACRO_CATEGORICAL_COLUMNS = [
    "scenario",
    "sector",
]

for column in MACRO_CATEGORICAL_COLUMNS:
    clean_macro_stress[column] = (
        clean_macro_stress[column]
        .astype("string")
        .str.strip()
    )

print("Macro stress categorical fields standardized.")

Macro stress categorical fields standardized.


In [53]:
# ---------------------------------------------------------------------------
# Standardize macro stress numeric fields
# ---------------------------------------------------------------------------

MACRO_NUMERIC_COLUMNS = [
    "gdp_shock_pp",
    "unemp_shock_pp",
    "rate_shock_pp",
    "credit_spread_bps",
    "base_pd",
    "stressed_pd",
    "pd_uplift_pp",
    "pd_multiplier",
    "base_lgd",
    "stressed_lgd",
    "total_ead",
    "expected_loss_base",
    "expected_loss_stress",
    "el_increase_pct",
]

for column in MACRO_NUMERIC_COLUMNS:
    clean_macro_stress[column] = pd.to_numeric(
        clean_macro_stress[column],
        errors="coerce",
    )

print("Macro stress numeric fields standardized.")

Macro stress numeric fields standardized.


In [54]:
# ---------------------------------------------------------------------------
# Validate scenario-sector grain
# ---------------------------------------------------------------------------

scenario_sector_counts = (
    clean_macro_stress
    .groupby(["scenario", "sector"])
    .size()
    .reset_index(name="record_count")
)

duplicate_scenario_sectors = (
    scenario_sector_counts
    .loc[scenario_sector_counts["record_count"] > 1]
)

print(
    f"Duplicate scenario-sector combinations: "
    f"{len(duplicate_scenario_sectors):,}"
)

display(duplicate_scenario_sectors)

Duplicate scenario-sector combinations: 0


,scenario,sector,record_count


In [55]:
# ---------------------------------------------------------------------------
# Validate stress-scenario relationships
# ---------------------------------------------------------------------------

clean_macro_stress["calculated_pd_uplift_pp"] = (
    clean_macro_stress["stressed_pd"]
    - clean_macro_stress["base_pd"]
)

clean_macro_stress["calculated_pd_multiplier"] = (
    clean_macro_stress["stressed_pd"]
    / clean_macro_stress["base_pd"]
)

clean_macro_stress["calculated_el_base"] = (
    clean_macro_stress["base_pd"]
    * clean_macro_stress["base_lgd"]
    * clean_macro_stress["total_ead"]
)

clean_macro_stress["calculated_el_stress"] = (
    clean_macro_stress["stressed_pd"]
    * clean_macro_stress["stressed_lgd"]
    * clean_macro_stress["total_ead"]
)

In [56]:
# ---------------------------------------------------------------------------
# Summarize stress-model consistency
# ---------------------------------------------------------------------------

stress_validation = pd.Series(
    {
        "pd_uplift_mismatches": (
            (
                clean_macro_stress["pd_uplift_pp"]
                - clean_macro_stress["calculated_pd_uplift_pp"]
            )
            .abs()
            .gt(1e-10)
            .sum()
        ),
        "pd_multiplier_mismatches": (
            (
                clean_macro_stress["pd_multiplier"]
                - clean_macro_stress["calculated_pd_multiplier"]
            )
            .abs()
            .gt(1e-10)
            .sum()
        ),
        "base_el_mismatches": (
            (
                clean_macro_stress["expected_loss_base"]
                - clean_macro_stress["calculated_el_base"]
            )
            .abs()
            .gt(0.01)
            .sum()
        ),
        "stress_el_mismatches": (
            (
                clean_macro_stress["expected_loss_stress"]
                - clean_macro_stress["calculated_el_stress"]
            )
            .abs()
            .gt(0.01)
            .sum()
        ),
    },
    name="count",
)

display(stress_validation.to_frame())

,count
pd_uplift_mismatches,50
pd_multiplier_mismatches,50
base_el_mismatches,60
stress_el_mismatches,60


In [57]:
# ---------------------------------------------------------------------------
# Inspect macro stress calculation fields
# ---------------------------------------------------------------------------

macro_formula_check = clean_macro_stress[
    [
        "scenario",
        "sector",
        "base_pd",
        "stressed_pd",
        "pd_uplift_pp",
        "pd_multiplier",
        "base_lgd",
        "stressed_lgd",
        "total_ead",
        "expected_loss_base",
        "expected_loss_stress",
        "el_increase_pct",
    ]
].copy()

display(macro_formula_check.head(20))

,scenario,sector,base_pd,stressed_pd,pd_uplift_pp,pd_multiplier,base_lgd,stressed_lgd,total_ead,expected_loss_base,expected_loss_stress,el_increase_pct
0,baseline,Financials,0.022308,0.022308,0.0000,1.000,0.45,0.4500,3.016669e+10,3.161070e+08,3.028334e+08,-4.20
1,baseline,Real_Estate,0.022656,0.022656,0.0000,1.000,0.40,0.4000,2.530543e+10,2.617922e+08,2.293309e+08,-12.40
2,baseline,Consumer,0.021524,0.021524,0.0000,1.000,0.55,0.5500,5.049910e+09,6.904421e+07,5.978082e+07,-13.42
3,baseline,Industrials,0.021915,0.021915,0.0000,1.000,0.50,0.5000,1.571215e+10,1.963758e+08,1.721656e+08,-12.33
4,baseline,Technology,0.023575,0.023575,0.0000,1.000,0.60,0.6000,1.277305e+10,2.140718e+08,1.806719e+08,-15.60
5,baseline,Energy,0.023161,0.023161,0.0000,1.000,0.45,0.4500,2.093550e+10,2.474053e+08,2.181981e+08,-11.81
6,baseline,Healthcare,0.022856,0.022856,0.0000,1.000,0.48,0.4800,1.004774e+10,1.182154e+08,1.102314e+08,-6.75
7,baseline,Utilities,0.022075,0.022075,0.0000,1.000,0.42,0.4200,2.206999e+10,2.358368e+08,2.046211e+08,-13.24
8,baseline,Retail,0.022869,0.022869,0.0000,1.000,0.58,0.5800,3.959381e+09,5.967482e+07,5.251826e+07,-11.99
9,baseline,Telecom,0.021696,0.021696,0.0000,1.000,0.50,0.5000,1.891039e+10,2.052627e+08,2.051427e+08,-0.06


In [58]:
# ---------------------------------------------------------------------------
# Inspect baseline scenario
# ---------------------------------------------------------------------------

baseline = clean_macro_stress.loc[
    clean_macro_stress["scenario"].eq("baseline")
].copy()

display(
    baseline[
        [
            "sector",
            "base_pd",
            "stressed_pd",
            "pd_uplift_pp",
            "pd_multiplier",
            "base_lgd",
            "stressed_lgd",
            "total_ead",
            "expected_loss_base",
            "expected_loss_stress",
            "el_increase_pct",
        ]
    ]
)

,sector,base_pd,stressed_pd,pd_uplift_pp,pd_multiplier,base_lgd,stressed_lgd,total_ead,expected_loss_base,expected_loss_stress,el_increase_pct
0,Financials,0.022308,0.022308,0.0,1.0,0.45,0.45,3.016669e+10,3.161070e+08,3.028334e+08,-4.20
1,Real_Estate,0.022656,0.022656,0.0,1.0,0.40,0.40,2.530543e+10,2.617922e+08,2.293309e+08,-12.40
2,Consumer,0.021524,0.021524,0.0,1.0,0.55,0.55,5.049910e+09,6.904421e+07,5.978082e+07,-13.42
3,Industrials,0.021915,0.021915,0.0,1.0,0.50,0.50,1.571215e+10,1.963758e+08,1.721656e+08,-12.33
4,Technology,0.023575,0.023575,0.0,1.0,0.60,0.60,1.277305e+10,2.140718e+08,1.806719e+08,-15.60
5,Energy,0.023161,0.023161,0.0,1.0,0.45,0.45,2.093550e+10,2.474053e+08,2.181981e+08,-11.81
6,Healthcare,0.022856,0.022856,0.0,1.0,0.48,0.48,1.004774e+10,1.182154e+08,1.102314e+08,-6.75
7,Utilities,0.022075,0.022075,0.0,1.0,0.42,0.42,2.206999e+10,2.358368e+08,2.046211e+08,-13.24
8,Retail,0.022869,0.022869,0.0,1.0,0.58,0.58,3.959381e+09,5.967482e+07,5.251826e+07,-11.99
9,Telecom,0.021696,0.021696,0.0,1.0,0.50,0.50,1.891039e+10,2.052627e+08,2.051427e+08,-0.06


In [59]:
# ---------------------------------------------------------------------------
# Compare stress outputs between baseline and severe scenarios
# ---------------------------------------------------------------------------

scenario_summary = (
    clean_macro_stress
    .groupby("scenario")
    .agg(
        average_base_pd=("base_pd", "mean"),
        average_stressed_pd=("stressed_pd", "mean"),
        average_base_lgd=("base_lgd", "mean"),
        average_stressed_lgd=("stressed_lgd", "mean"),
        total_ead=("total_ead", "sum"),
        total_expected_loss_base=("expected_loss_base", "sum"),
        total_expected_loss_stress=("expected_loss_stress", "sum"),
    )
)

display(scenario_summary)

,average_base_pd,average_stressed_pd,average_base_lgd,average_stressed_lgd,total_ead,total_expected_loss_base,total_expected_loss_stress
scenario,,,,,,,
adverse,0.022464,0.033556,0.493,0.5222,1.649302e+11,1.923786e+09,2.784685e+09
baseline,0.022464,0.022464,0.493,0.4930,1.649302e+11,1.923786e+09,1.735494e+09
covid_like,0.022464,0.056768,0.493,0.5597,1.649302e+11,1.923786e+09,5.121992e+09
gfc_like,0.022464,0.040583,0.493,0.5347,1.649302e+11,1.923786e+09,3.466313e+09
mild,0.022464,0.026908,0.493,0.5055,1.649302e+11,1.923786e+09,2.145252e+09
severe,0.022464,0.044744,0.493,0.5430,1.649302e+11,1.923786e+09,3.893765e+09


In [60]:
# ---------------------------------------------------------------------------
# Inspect macro stress calculations at row level
# ---------------------------------------------------------------------------

macro_formula_check = clean_macro_stress[
    [
        "scenario",
        "sector",
        "base_pd",
        "stressed_pd",
        "base_lgd",
        "stressed_lgd",
        "total_ead",
        "expected_loss_base",
        "expected_loss_stress",
    ]
].copy()

macro_formula_check["calculated_base_el"] = (
    macro_formula_check["base_pd"]
    * macro_formula_check["base_lgd"]
    * macro_formula_check["total_ead"]
)

macro_formula_check["calculated_stress_el"] = (
    macro_formula_check["stressed_pd"]
    * macro_formula_check["stressed_lgd"]
    * macro_formula_check["total_ead"]
)

display(
    macro_formula_check.head(20)
)

,scenario,sector,base_pd,stressed_pd,base_lgd,stressed_lgd,total_ead,expected_loss_base,expected_loss_stress,calculated_base_el,calculated_stress_el
0,baseline,Financials,0.022308,0.022308,0.45,0.4500,3.016669e+10,3.161070e+08,3.028334e+08,3.028314e+08,3.028314e+08
1,baseline,Real_Estate,0.022656,0.022656,0.40,0.4000,2.530543e+10,2.617922e+08,2.293309e+08,2.293279e+08,2.293279e+08
2,baseline,Consumer,0.021524,0.021524,0.55,0.5500,5.049910e+09,6.904421e+07,5.978082e+07,5.978185e+07,5.978185e+07
3,baseline,Industrials,0.021915,0.021915,0.50,0.5000,1.571215e+10,1.963758e+08,1.721656e+08,1.721659e+08,1.721659e+08
4,baseline,Technology,0.023575,0.023575,0.60,0.6000,1.277305e+10,2.140718e+08,1.806719e+08,1.806748e+08,1.806748e+08
5,baseline,Energy,0.023161,0.023161,0.45,0.4500,2.093550e+10,2.474053e+08,2.181981e+08,2.181992e+08,2.181992e+08
6,baseline,Healthcare,0.022856,0.022856,0.48,0.4800,1.004774e+10,1.182154e+08,1.102314e+08,1.102326e+08,1.102326e+08
7,baseline,Utilities,0.022075,0.022075,0.42,0.4200,2.206999e+10,2.358368e+08,2.046211e+08,2.046219e+08,2.046219e+08
8,baseline,Retail,0.022869,0.022869,0.58,0.5800,3.959381e+09,5.967482e+07,5.251826e+07,5.251730e+07,5.251730e+07
9,baseline,Telecom,0.021696,0.021696,0.50,0.5000,1.891039e+10,2.052627e+08,2.051427e+08,2.051399e+08,2.051399e+08


In [61]:
# ---------------------------------------------------------------------------
# Compare supplied and independently calculated stress EL
# ---------------------------------------------------------------------------

macro_formula_check["base_el_difference"] = (
    macro_formula_check["expected_loss_base"]
    - macro_formula_check["calculated_base_el"]
)

macro_formula_check["stress_el_difference"] = (
    macro_formula_check["expected_loss_stress"]
    - macro_formula_check["calculated_stress_el"]
)

display(
    macro_formula_check[
        [
            "scenario",
            "sector",
            "expected_loss_base",
            "calculated_base_el",
            "base_el_difference",
            "expected_loss_stress",
            "calculated_stress_el",
            "stress_el_difference",
        ]
    ].head(20)
)

,scenario,sector,expected_loss_base,calculated_base_el,base_el_difference,expected_loss_stress,calculated_stress_el,stress_el_difference
0,baseline,Financials,3.161070e+08,3.028314e+08,1.327562e+07,3.028334e+08,3.028314e+08,2064.760352
1,baseline,Real_Estate,2.617922e+08,2.293279e+08,3.246426e+07,2.293309e+08,2.293279e+08,2906.326509
2,baseline,Consumer,6.904421e+07,5.978185e+07,9.262365e+06,5.978082e+07,5.978185e+07,-1023.854834
3,baseline,Industrials,1.963758e+08,1.721659e+08,2.420985e+07,1.721656e+08,1.721659e+08,-277.439722
4,baseline,Technology,2.140718e+08,1.806748e+08,3.339703e+07,1.806719e+08,1.806748e+08,-2851.336417
5,baseline,Energy,2.474053e+08,2.181992e+08,2.920618e+07,2.181981e+08,2.181992e+08,-1103.277098
6,baseline,Healthcare,1.182154e+08,1.102326e+08,7.982805e+06,1.102314e+08,1.102326e+08,-1161.681814
7,baseline,Utilities,2.358368e+08,2.046219e+08,3.121492e+07,2.046211e+08,2.046219e+08,-788.807512
8,baseline,Retail,5.967482e+07,5.251730e+07,7.157519e+06,5.251826e+07,5.251730e+07,953.175899
9,baseline,Telecom,2.052627e+08,2.051399e+08,1.227988e+05,2.051427e+08,2.051399e+08,2802.847636


### Macro Stress Expected-Loss Validation — Finding

The supplied `expected_loss_stress` values closely match independently
calculated:

**stressed_pd × stressed_lgd × total_ead**

at the scenario-sector level, with only small numerical differences.

However, the supplied `expected_loss_base` values do not equal:

**base_pd × base_lgd × total_ead**

for the scenario-sector rows.

**Interpretation:** The source dataset uses a different calculation or
aggregation method for the base expected-loss field. The exact construction
cannot be established from the available columns alone.

**Decision:** Preserve the source `expected_loss_base` and
`expected_loss_stress` values unchanged. Do not replace them with our
reconstructed values. Independently calculated values will be retained only
as validation/reference fields.

This distinction will be documented in the final methodology.

In [62]:
# ---------------------------------------------------------------------------
# Inspect stress-output formulas
# ---------------------------------------------------------------------------

macro_output_check = clean_macro_stress[
    [
        "scenario",
        "sector",
        "base_pd",
        "stressed_pd",
        "pd_uplift_pp",
        "pd_multiplier",
        "base_lgd",
        "stressed_lgd",
        "el_increase_pct",
        "expected_loss_base",
        "expected_loss_stress",
    ]
].copy()

macro_output_check["pd_difference"] = (
    macro_output_check["stressed_pd"]
    - macro_output_check["base_pd"]
)

macro_output_check["pd_ratio"] = (
    macro_output_check["stressed_pd"]
    / macro_output_check["base_pd"]
)

macro_output_check["el_increase_calculated_pct"] = (
    (
        macro_output_check["expected_loss_stress"]
        / macro_output_check["expected_loss_base"]
    ) - 1
) * 100

display(macro_output_check.head(20))

,scenario,sector,base_pd,stressed_pd,pd_uplift_pp,pd_multiplier,base_lgd,stressed_lgd,el_increase_pct,expected_loss_base,expected_loss_stress,pd_difference,pd_ratio,el_increase_calculated_pct
0,baseline,Financials,0.022308,0.022308,0.0000,1.000,0.45,0.4500,-4.20,3.161070e+08,3.028334e+08,0.000000,1.000000,-4.199072
1,baseline,Real_Estate,0.022656,0.022656,0.0000,1.000,0.40,0.4000,-12.40,2.617922e+08,2.293309e+08,0.000000,1.000000,-12.399665
2,baseline,Consumer,0.021524,0.021524,0.0000,1.000,0.55,0.5500,-13.42,6.904421e+07,5.978082e+07,0.000000,1.000000,-13.416604
3,baseline,Industrials,0.021915,0.021915,0.0000,1.000,0.50,0.5000,-12.33,1.963758e+08,1.721656e+08,0.000000,1.000000,-12.328471
4,baseline,Technology,0.023575,0.023575,0.0000,1.000,0.60,0.6000,-15.60,2.140718e+08,1.806719e+08,0.000000,1.000000,-15.602184
5,baseline,Energy,0.023161,0.023161,0.0000,1.000,0.45,0.4500,-11.81,2.474053e+08,2.181981e+08,0.000000,1.000000,-11.805438
6,baseline,Healthcare,0.022856,0.022856,0.0000,1.000,0.48,0.4800,-6.75,1.182154e+08,1.102314e+08,0.000000,1.000000,-6.753744
7,baseline,Utilities,0.022075,0.022075,0.0000,1.000,0.42,0.4200,-13.24,2.358368e+08,2.046211e+08,0.000000,1.000000,-13.236148
8,baseline,Retail,0.022869,0.022869,0.0000,1.000,0.58,0.5800,-11.99,5.967482e+07,5.251826e+07,0.000000,1.000000,-11.992605
9,baseline,Telecom,0.021696,0.021696,0.0000,1.000,0.50,0.5000,-0.06,2.052627e+08,2.051427e+08,0.000000,1.000000,-0.058460


In [63]:
# ---------------------------------------------------------------------------
# Compare supplied and reconstructed stress metrics
# ---------------------------------------------------------------------------

stress_metric_comparison = pd.DataFrame(
    {
        "pd_uplift_max_abs_diff": (
            (
                macro_output_check["pd_uplift_pp"]
                - macro_output_check["pd_difference"]
            )
            .abs()
            .max()
        ),
        "pd_multiplier_max_abs_diff": (
            (
                macro_output_check["pd_multiplier"]
                - macro_output_check["pd_ratio"]
            )
            .abs()
            .max()
        ),
        "el_increase_max_abs_diff_pct": (
            (
                macro_output_check["el_increase_pct"]
                - macro_output_check["el_increase_calculated_pct"]
            )
            .abs()
            .max()
        ),
    },
    index=["maximum_difference"],
).T

display(stress_metric_comparison)

,maximum_difference
pd_uplift_max_abs_diff,4.065237
pd_multiplier_max_abs_diff,0.000508
el_increase_max_abs_diff_pct,0.004989


### Macro Stress Output Validation — Final Finding

The macro stress scenario outputs were independently tested.

**PD uplift**

The supplied `pd_uplift_pp` field is consistent with:

**(stressed_pd − base_pd) × 100**

The apparent discrepancy in the initial validation was caused by comparing a
percentage-point value with a decimal probability difference.

**PD multiplier**

The supplied `pd_multiplier` is consistent with:

**stressed_pd / base_pd**

with only a very small numerical difference.

**Expected-loss increase**

The supplied `el_increase_pct` is consistent with:

**((expected_loss_stress / expected_loss_base) − 1) × 100**

with only a very small numerical difference.

**Stress expected loss**

The supplied `expected_loss_stress` closely matches:

**stressed_pd × stressed_lgd × total_ead**

subject to small numerical precision differences.

**Base expected loss**

The supplied `expected_loss_base` does not equal:

**base_pd × base_lgd × total_ead**

at the scenario-sector level. The available fields do not provide enough
information to establish the source calculation. The source value will
therefore be preserved as a benchmark rather than overwritten.

**Decision:** No source stress-output values will be modified during cleaning.

In [64]:
clean_macro_stress["pd_uplift"] = (
    clean_macro_stress["stressed_pd"]
    - clean_macro_stress["base_pd"]
)

In [65]:
# ---------------------------------------------------------------------------
# Create standardized analytical PD uplift
# ---------------------------------------------------------------------------

clean_macro_stress["pd_uplift"] = (
    clean_macro_stress["stressed_pd"]
    - clean_macro_stress["base_pd"]
)

print("Standardized decimal PD uplift created.")

Standardized decimal PD uplift created.


In [66]:
display(
    clean_macro_stress[
        [
            "scenario",
            "sector",
            "base_pd",
            "stressed_pd",
            "pd_uplift",
            "pd_uplift_pp",
            "pd_multiplier",
        ]
    ].head(10)
)

,scenario,sector,base_pd,stressed_pd,pd_uplift,pd_uplift_pp,pd_multiplier
0,baseline,Financials,0.022308,0.022308,0.0,0.0,1.0
1,baseline,Real_Estate,0.022656,0.022656,0.0,0.0,1.0
2,baseline,Consumer,0.021524,0.021524,0.0,0.0,1.0
3,baseline,Industrials,0.021915,0.021915,0.0,0.0,1.0
4,baseline,Technology,0.023575,0.023575,0.0,0.0,1.0
5,baseline,Energy,0.023161,0.023161,0.0,0.0,1.0
6,baseline,Healthcare,0.022856,0.022856,0.0,0.0,1.0
7,baseline,Utilities,0.022075,0.022075,0.0,0.0,1.0
8,baseline,Retail,0.022869,0.022869,0.0,0.0,1.0
9,baseline,Telecom,0.021696,0.021696,0.0,0.0,1.0


In [67]:
# ---------------------------------------------------------------------------
# Remove temporary validation columns
# ---------------------------------------------------------------------------

TEMPORARY_COLUMNS = [
    "from_score",
    "to_score",
    "calculated_notches_moved",
    "ecl_calculated",
    "el_difference",
    "el_relative_difference",
    "lgd_calculated",
    "loss_calculated",
    "lgd_difference",
    "loss_difference",
    "calculated_base_el",
    "calculated_stress_el",
    "base_el_difference",
    "stress_el_difference",
    "pd_difference",
    "pd_ratio",
    "el_increase_calculated_pct",
]

clean_macro_stress = clean_macro_stress.drop(
    columns=[
        column
        for column in TEMPORARY_COLUMNS
        if column in clean_macro_stress.columns
    ]
)

print("Temporary audit columns removed.")

Temporary audit columns removed.


In [68]:
# ---------------------------------------------------------------------------
# Save cleaned macro stress scenarios
# ---------------------------------------------------------------------------

CLEAN_MACRO_STRESS_FILE = (
    PROCESSED_DATA_DIR
    / "macro_stress_scenarios_clean.csv"
)

clean_macro_stress.to_csv(
    CLEAN_MACRO_STRESS_FILE,
    index=False,
)

print(
    f"Saved cleaned macro stress scenarios to:\n"
    f"{CLEAN_MACRO_STRESS_FILE}"
)

Saved cleaned macro stress scenarios to:
D:\Bank_sense_2.0\data\processed\macro_stress_scenarios_clean.csv


In [69]:
# ---------------------------------------------------------------------------
# Verify processed output
# ---------------------------------------------------------------------------

if not CLEAN_MACRO_STRESS_FILE.exists():
    raise FileNotFoundError(
        f"Processed file was not created: "
        f"{CLEAN_MACRO_STRESS_FILE}"
    )

print(
    f"Verified: {CLEAN_MACRO_STRESS_FILE.name}"
)

Verified: macro_stress_scenarios_clean.csv


In [70]:
# ---------------------------------------------------------------------------
# Load raw portfolio metrics
# ---------------------------------------------------------------------------

PORTFOLIO_METRICS_FILE = (
    RAW_DATA_DIR / "portfolio_metrics.csv"
)

if not PORTFOLIO_METRICS_FILE.exists():
    raise FileNotFoundError(
        f"Portfolio metrics file not found: "
        f"{PORTFOLIO_METRICS_FILE}"
    )

portfolio_metrics = pd.read_csv(
    PORTFOLIO_METRICS_FILE,
    low_memory=False,
)

print(
    f"Loaded {len(portfolio_metrics):,} portfolio-metric records "
    f"with {len(portfolio_metrics.columns)} columns."
)

Loaded 120 portfolio-metric records with 16 columns.


In [71]:
# ---------------------------------------------------------------------------
# Create cleaning workspace
# ---------------------------------------------------------------------------

clean_portfolio_metrics = portfolio_metrics.copy()

print("Portfolio-metrics cleaning workspace created.")

Portfolio-metrics cleaning workspace created.


In [72]:
# ---------------------------------------------------------------------------
# Initial portfolio-metrics profile
# ---------------------------------------------------------------------------

portfolio_metrics_profile = pd.DataFrame(
    {
        "column": clean_portfolio_metrics.columns,
        "dtype": clean_portfolio_metrics.dtypes.astype(str).values,
        "missing_count": clean_portfolio_metrics.isna().sum().values,
        "unique_values": clean_portfolio_metrics.nunique(
            dropna=False
        ).values,
    }
)

display(portfolio_metrics_profile)

,column,dtype,missing_count,unique_values
0,date,str,0,120
1,n_active_loans,int64,0,119
2,total_ead,float64,0,120
3,total_el,float64,0,120
4,total_rwa,float64,0,120
5,el_rate,float64,0,116
6,avg_pd,float64,0,116
7,avg_lgd,float64,0,29
8,var_99,float64,0,120
9,cvar_995,float64,0,120


In [73]:
# ---------------------------------------------------------------------------
# Initial portfolio-metrics records
# ---------------------------------------------------------------------------

display(
    clean_portfolio_metrics.head(10)
)

,date,n_active_loans,total_ead,total_el,total_rwa,el_rate,avg_pd,avg_lgd,var_99,cvar_995,sector_hhi,new_defaults,gdp_growth,unemployment,policy_rate,credit_spread_bps
0,2015-01-01,482,1.697996e+09,2.574764e+07,3.411562e+08,0.015164,0.023472,0.5378,5.999200e+07,7.544058e+07,0.1312,0,2.591,5.319,3.0,154.0
1,2015-02-01,956,3.132179e+09,4.021254e+07,5.328161e+08,0.012839,0.021194,0.5407,9.369521e+07,1.178227e+08,0.1282,1,2.864,5.010,3.0,141.2
2,2015-03-01,1415,4.428744e+09,5.298450e+07,7.020446e+08,0.011964,0.021226,0.5437,1.234539e+08,1.552446e+08,0.1245,1,2.866,4.962,3.0,139.9
3,2015-04-01,1898,6.060836e+09,7.143560e+07,9.465217e+08,0.011786,0.021117,0.5422,1.664449e+08,2.093063e+08,0.1259,2,2.574,5.096,3.0,145.2
4,2015-05-01,2378,7.439622e+09,8.812599e+07,1.167669e+09,0.011845,0.021781,0.5433,2.053335e+08,2.582091e+08,0.1250,0,2.586,5.261,3.0,147.4
5,2015-06-01,2840,8.882164e+09,1.022002e+08,1.354153e+09,0.011506,0.021398,0.5449,2.381265e+08,2.994466e+08,0.1243,8,2.320,5.303,3.0,138.0
6,2015-07-01,3312,1.053701e+10,1.200292e+08,1.590387e+09,0.011391,0.021685,0.5450,2.796681e+08,3.516856e+08,0.1242,8,2.601,5.280,3.0,135.7
7,2015-08-01,3764,1.201845e+10,1.428131e+08,1.892274e+09,0.011883,0.022736,0.5454,3.327545e+08,4.184424e+08,0.1249,7,2.387,5.450,3.0,133.9
8,2015-09-01,4217,1.374942e+10,1.619882e+08,2.146343e+09,0.011781,0.022574,0.5454,3.774324e+08,4.746253e+08,0.1244,11,2.270,5.374,3.0,137.8
9,2015-10-01,4649,1.519604e+10,1.750542e+08,2.319468e+09,0.011520,0.022231,0.5456,4.078763e+08,5.129089e+08,0.1256,4,2.402,5.417,3.0,140.4


In [74]:
# ---------------------------------------------------------------------------
# Standardize portfolio date
# ---------------------------------------------------------------------------

clean_portfolio_metrics["date"] = pd.to_datetime(
    clean_portfolio_metrics["date"],
    errors="coerce",
)

print("Portfolio date standardized.")

Portfolio date standardized.


In [75]:
# ---------------------------------------------------------------------------
# Validate portfolio date field
# ---------------------------------------------------------------------------

portfolio_date_check = pd.Series(
    {
        "missing_dates": int(
            clean_portfolio_metrics["date"].isna().sum()
        ),
        "minimum_date": clean_portfolio_metrics["date"].min(),
        "maximum_date": clean_portfolio_metrics["date"].max(),
        "unique_dates": clean_portfolio_metrics["date"].nunique(),
    },
    name="value",
)

display(portfolio_date_check.to_frame())

,value
missing_dates,0
minimum_date,2015-01-01 00:00:00
maximum_date,2024-12-01 00:00:00
unique_dates,120


In [76]:
# ---------------------------------------------------------------------------
# Validate monthly portfolio grain
# ---------------------------------------------------------------------------

duplicate_dates = (
    clean_portfolio_metrics["date"]
    .duplicated()
    .sum()
)

print(
    f"Duplicate portfolio dates: {duplicate_dates:,}"
)

Duplicate portfolio dates: 0


In [77]:
# ---------------------------------------------------------------------------
# Validate monthly continuity
# ---------------------------------------------------------------------------

portfolio_dates = (
    clean_portfolio_metrics["date"]
    .sort_values()
    .reset_index(drop=True)
)

month_gaps = (
    portfolio_dates
    .dt.to_period("M")
    .astype("int64")
    .diff()
    .dropna()
)

month_gap_summary = (
    month_gaps
    .value_counts()
    .sort_index()
    .rename("count")
    .to_frame()
)

display(month_gap_summary)

,count
date,
1.0,119


In [78]:
# ---------------------------------------------------------------------------
# Validate portfolio expected-loss rate
# ---------------------------------------------------------------------------

clean_portfolio_metrics["calculated_el_rate"] = (
    clean_portfolio_metrics["total_el"]
    / clean_portfolio_metrics["total_ead"]
)

clean_portfolio_metrics["el_rate_difference"] = (
    clean_portfolio_metrics["el_rate"]
    - clean_portfolio_metrics["calculated_el_rate"]
)

display(
    clean_portfolio_metrics[
        [
            "date",
            "total_ead",
            "total_el",
            "el_rate",
            "calculated_el_rate",
            "el_rate_difference",
        ]
    ].head(10)
)

,date,total_ead,total_el,el_rate,calculated_el_rate,el_rate_difference
0,2015-01-01,1.697996e+09,2.574764e+07,0.015164,0.015164,4.539825e-07
1,2015-02-01,3.132179e+09,4.021254e+07,0.012839,0.012839,4.798901e-07
2,2015-03-01,4.428744e+09,5.298450e+07,0.011964,0.011964,2.253993e-07
3,2015-04-01,6.060836e+09,7.143560e+07,0.011786,0.011786,-4.262076e-07
4,2015-05-01,7.439622e+09,8.812599e+07,0.011845,0.011845,-4.918953e-07
5,2015-06-01,8.882164e+09,1.022002e+08,0.011506,0.011506,-2.281220e-07
6,2015-07-01,1.053701e+10,1.200292e+08,0.011391,0.011391,-2.065399e-07
7,2015-08-01,1.201845e+10,1.428131e+08,0.011883,0.011883,1.768167e-07
8,2015-09-01,1.374942e+10,1.619882e+08,0.011781,0.011781,-4.538403e-07
9,2015-10-01,1.519604e+10,1.750542e+08,0.011520,0.011520,2.743280e-07


In [79]:
# ---------------------------------------------------------------------------
# Expected-loss rate validation
# ---------------------------------------------------------------------------

el_rate_validation = (
    clean_portfolio_metrics["el_rate_difference"]
    .abs()
    .describe()
)

display(
    el_rate_validation.to_frame("absolute_difference")
)

,absolute_difference
count,1.200000e+02
mean,2.767343e-07
std,1.460114e-07
min,5.675214e-09
25%,1.608640e-07
50%,2.878703e-07
75%,4.071288e-07
max,4.990457e-07


### Portfolio Expected-Loss Rate Validation

The supplied `el_rate` field was independently reconstructed as:

**EL Rate = Total Expected Loss / Total EAD**

Across all 120 monthly portfolio observations:

- Mean absolute difference: **2.77 × 10⁻⁷**
- Median absolute difference: **2.88 × 10⁻⁷**
- Maximum absolute difference: **4.99 × 10⁻⁷**

**Conclusion:** The supplied `el_rate` is internally consistent with
`total_el / total_ead`, with only negligible numerical differences.
No correction is required.

In [80]:
display(month_gap_summary)

,count
date,
1.0,119


In [81]:
# ---------------------------------------------------------------------------
# Validate monthly portfolio continuity
# ---------------------------------------------------------------------------

display(month_gap_summary)

,count
date,
1.0,119


### Portfolio Time-Series Continuity — Finding

The `portfolio_metrics` dataset contains **120 monthly observations**.

All **119 consecutive observations** have exactly a one-month interval.

- One-month gaps: **119**
- Other gaps: **0**

**Conclusion:** The portfolio time series is continuous from its first to its
last observation. No date-gap correction or interpolation is required.

In [82]:
# ---------------------------------------------------------------------------
# Portfolio risk-metric range checks
# ---------------------------------------------------------------------------

portfolio_range_checks = pd.DataFrame(
    {
        "check": [
            "n_active_loans < 0",
            "total_ead <= 0",
            "total_el < 0",
            "el_rate < 0",
            "el_rate > 1",
            "avg_pd < 0",
            "avg_pd > 1",
            "avg_lgd < 0",
            "avg_lgd > 1",
            "var_99 < 0",
            "cvar_995 < 0",
            "sector_hhi < 0",
            "sector_hhi > 1",
            "new_defaults < 0",
            "unemployment < 0",
            "policy_rate < 0",
            "credit_spread_bps < 0",
        ],
        "count": [
            clean_portfolio_metrics["n_active_loans"].lt(0).sum(),
            clean_portfolio_metrics["total_ead"].le(0).sum(),
            clean_portfolio_metrics["total_el"].lt(0).sum(),
            clean_portfolio_metrics["el_rate"].lt(0).sum(),
            clean_portfolio_metrics["el_rate"].gt(1).sum(),
            clean_portfolio_metrics["avg_pd"].lt(0).sum(),
            clean_portfolio_metrics["avg_pd"].gt(1).sum(),
            clean_portfolio_metrics["avg_lgd"].lt(0).sum(),
            clean_portfolio_metrics["avg_lgd"].gt(1).sum(),
            clean_portfolio_metrics["var_99"].lt(0).sum(),
            clean_portfolio_metrics["cvar_995"].lt(0).sum(),
            clean_portfolio_metrics["sector_hhi"].lt(0).sum(),
            clean_portfolio_metrics["sector_hhi"].gt(1).sum(),
            clean_portfolio_metrics["new_defaults"].lt(0).sum(),
            clean_portfolio_metrics["unemployment"].lt(0).sum(),
            clean_portfolio_metrics["policy_rate"].lt(0).sum(),
            clean_portfolio_metrics["credit_spread_bps"].lt(0).sum(),
        ],
    }
)

display(portfolio_range_checks)

,check,count
0,n_active_loans < 0,0
1,total_ead <= 0,0
2,total_el < 0,0
3,el_rate < 0,0
4,el_rate > 1,0
5,avg_pd < 0,0
6,avg_pd > 1,0
7,avg_lgd < 0,0
8,avg_lgd > 1,0
9,var_99 < 0,0


In [83]:
# ---------------------------------------------------------------------------
# Validate VaR / CVaR ordering
# ---------------------------------------------------------------------------

var_cvar_check = pd.Series(
    {
        "cvar_below_var": (
            clean_portfolio_metrics["cvar_995"]
            < clean_portfolio_metrics["var_99"]
        ).sum(),
        "cvar_equal_var": (
            clean_portfolio_metrics["cvar_995"]
            == clean_portfolio_metrics["var_99"]
        ).sum(),
        "cvar_above_var": (
            clean_portfolio_metrics["cvar_995"]
            > clean_portfolio_metrics["var_99"]
        ).sum(),
    },
    name="count",
)

display(var_cvar_check.to_frame())

,count
cvar_below_var,0
cvar_equal_var,0
cvar_above_var,120


### Portfolio Risk-Metric Validation — Finding

All defined portfolio-level range checks returned **zero violations**.

The dataset also satisfies the expected ordering between its tail-risk measures:

**CVaR 99.5% > VaR 99%**

for all **120 monthly observations**.

- CVaR below VaR: **0**
- CVaR equal to VaR: **0**
- CVaR above VaR: **120**

**Conclusion:** The portfolio risk metrics are within their expected numeric
ranges, and the supplied VaR/CVaR measures show consistent tail-risk ordering.

In [84]:
# ---------------------------------------------------------------------------
# Portfolio risk-metric distribution summary
# ---------------------------------------------------------------------------

portfolio_distribution_summary = (
    clean_portfolio_metrics[
        [
            "avg_pd",
            "avg_lgd",
            "el_rate",
            "sector_hhi",
            "var_99",
            "cvar_995",
            "gdp_growth",
            "unemployment",
            "policy_rate",
            "credit_spread_bps",
        ]
    ]
    .describe()
    .T
)

display(portfolio_distribution_summary)

,count,mean,std,min,25%,50%,75%,max
avg_pd,120.0,2.233495e-02,9.456906e-04,2.111700e-02,2.139350e-02,2.228900e-02,2.341050e-02,2.371600e-02
avg_lgd,120.0,5.460525e-01,1.179799e-03,5.378000e-01,5.457000e-01,5.464000e-01,5.467000e-01,5.473000e-01
el_rate,120.0,1.174054e-02,5.603033e-04,1.102700e-02,1.130250e-02,1.163100e-02,1.211225e-02,1.516400e-02
sector_hhi,120.0,1.245217e-01,8.997837e-04,1.229000e-01,1.240000e-01,1.244000e-01,1.249000e-01,1.312000e-01
var_99,120.0,1.629607e+09,6.672339e+08,5.999200e+07,1.219220e+09,1.808366e+09,2.225764e+09,2.322674e+09
cvar_995,120.0,2.049249e+09,8.390538e+08,7.544058e+07,1.533183e+09,2.274040e+09,2.798922e+09,2.920788e+09
gdp_growth,120.0,2.323342e+00,1.422392e+00,-4.257000e+00,2.284250e+00,2.588500e+00,2.871000e+00,4.490000e+00
unemployment,120.0,5.804292e+00,1.748261e+00,4.168000e+00,4.799250e+00,5.349500e+00,5.970250e+00,1.227900e+01
policy_rate,120.0,3.003583e+00,9.664359e-01,1.500000e+00,3.000000e+00,3.000000e+00,3.062500e+00,5.000000e+00
credit_spread_bps,120.0,1.423050e+02,3.514140e+01,1.004000e+02,1.225000e+02,1.351000e+02,1.438250e+02,3.073000e+02


In [85]:
# ---------------------------------------------------------------------------
# Portfolio metric endpoints
# ---------------------------------------------------------------------------

portfolio_endpoints = pd.DataFrame(
    {
        "metric": [
            "avg_pd",
            "avg_lgd",
            "el_rate",
            "sector_hhi",
            "gdp_growth",
            "unemployment",
            "policy_rate",
            "credit_spread_bps",
        ],
        "minimum": [
            clean_portfolio_metrics["avg_pd"].min(),
            clean_portfolio_metrics["avg_lgd"].min(),
            clean_portfolio_metrics["el_rate"].min(),
            clean_portfolio_metrics["sector_hhi"].min(),
            clean_portfolio_metrics["gdp_growth"].min(),
            clean_portfolio_metrics["unemployment"].min(),
            clean_portfolio_metrics["policy_rate"].min(),
            clean_portfolio_metrics["credit_spread_bps"].min(),
        ],
        "maximum": [
            clean_portfolio_metrics["avg_pd"].max(),
            clean_portfolio_metrics["avg_lgd"].max(),
            clean_portfolio_metrics["el_rate"].max(),
            clean_portfolio_metrics["sector_hhi"].max(),
            clean_portfolio_metrics["gdp_growth"].max(),
            clean_portfolio_metrics["unemployment"].max(),
            clean_portfolio_metrics["policy_rate"].max(),
            clean_portfolio_metrics["credit_spread_bps"].max(),
        ],
    }
)

display(portfolio_endpoints)

,metric,minimum,maximum
0,avg_pd,0.021117,0.023716
1,avg_lgd,0.537800,0.547300
2,el_rate,0.011027,0.015164
3,sector_hhi,0.122900,0.131200
4,gdp_growth,-4.257000,4.490000
5,unemployment,4.168000,12.279000
6,policy_rate,1.500000,5.000000
7,credit_spread_bps,100.400000,307.300000


### Portfolio Metric Distribution — Finding

The monthly portfolio risk metrics remain within stable ranges:

- Average PD: **2.11%–2.37%**
- Average LGD: **53.78%–54.73%**
- Expected-loss rate: **1.10%–1.52%**
- Sector HHI: **0.1229–0.1312**

Macro variables show substantially greater variation, including GDP growth from
**−4.257% to 4.49%**, unemployment from **4.17% to 12.28%**, and credit spreads
from **100.4 to 307.3 bps**.

**Conclusion:** No obvious range violations or implausible values were found.
The variation in macro variables provides useful input for later portfolio
stress-testing analysis.

In [86]:
# ---------------------------------------------------------------------------
# Final portfolio-metrics missing-value check
# ---------------------------------------------------------------------------

portfolio_missing = (
    clean_portfolio_metrics
    .isna()
    .sum()
    .sort_values(ascending=False)
)

display(
    portfolio_missing
    .rename("missing_count")
    .to_frame()
)

,missing_count
date,0
n_active_loans,0
total_ead,0
total_el,0
total_rwa,0
el_rate,0
avg_pd,0
avg_lgd,0
var_99,0
cvar_995,0


In [87]:
# ---------------------------------------------------------------------------
# Remove temporary validation columns
# ---------------------------------------------------------------------------

TEMPORARY_PORTFOLIO_COLUMNS = [
    "calculated_el_rate",
    "el_rate_difference",
]

clean_portfolio_metrics = clean_portfolio_metrics.drop(
    columns=[
        column
        for column in TEMPORARY_PORTFOLIO_COLUMNS
        if column in clean_portfolio_metrics.columns
    ]
)

print("Temporary portfolio validation columns removed.")

Temporary portfolio validation columns removed.


In [88]:
# ---------------------------------------------------------------------------
# Save cleaned portfolio metrics
# ---------------------------------------------------------------------------

CLEAN_PORTFOLIO_METRICS_FILE = (
    PROCESSED_DATA_DIR / "portfolio_metrics_clean.csv"
)

clean_portfolio_metrics.to_csv(
    CLEAN_PORTFOLIO_METRICS_FILE,
    index=False,
)

print(
    f"Saved cleaned portfolio metrics to:\n"
    f"{CLEAN_PORTFOLIO_METRICS_FILE}"
)

Saved cleaned portfolio metrics to:
D:\Bank_sense_2.0\data\processed\portfolio_metrics_clean.csv


In [89]:
if not CLEAN_PORTFOLIO_METRICS_FILE.exists():
    raise FileNotFoundError(
        f"Processed file was not created: "
        f"{CLEAN_PORTFOLIO_METRICS_FILE}"
    )

print(
    f"Verified: {CLEAN_PORTFOLIO_METRICS_FILE.name}"
)

Verified: portfolio_metrics_clean.csv


In [90]:
# ---------------------------------------------------------------------------
# Load raw vintage analysis data
# ---------------------------------------------------------------------------

VINTAGE_FILE = RAW_DATA_DIR / "vintage_analysis.csv"

if not VINTAGE_FILE.exists():
    raise FileNotFoundError(
        f"Vintage analysis file not found: {VINTAGE_FILE}"
    )

vintage_analysis = pd.read_csv(
    VINTAGE_FILE,
    low_memory=False,
)

print(
    f"Loaded {len(vintage_analysis):,} vintage records "
    f"with {len(vintage_analysis.columns)} columns."
)

Loaded 2,160 vintage records with 9 columns.


In [91]:
# ---------------------------------------------------------------------------
# Create cleaning workspace
# ---------------------------------------------------------------------------

clean_vintage_analysis = vintage_analysis.copy()

print("Vintage-analysis cleaning workspace created.")

Vintage-analysis cleaning workspace created.


In [92]:
# ---------------------------------------------------------------------------
# Initial vintage-analysis profile
# ---------------------------------------------------------------------------

vintage_profile = pd.DataFrame(
    {
        "column": clean_vintage_analysis.columns,
        "dtype": clean_vintage_analysis.dtypes.astype(str).values,
        "missing_count": clean_vintage_analysis.isna().sum().values,
        "unique_values": clean_vintage_analysis.nunique(
            dropna=False
        ).values,
    }
)

display(vintage_profile)

,column,dtype,missing_count,unique_values
0,vintage,str,0,36
1,months_on_books,int64,0,60
2,n_loans_originated,int64,0,32
3,n_active,int64,0,734
4,n_defaulted_cumulative,int64,0,280
5,cumulative_default_rate,float64,0,1614
6,marginal_default_rate,float64,0,1185
7,avg_pd_at_origination,float64,0,36
8,avg_credit_score,float64,0,23


In [93]:
# ---------------------------------------------------------------------------
# Initial vintage-analysis records
# ---------------------------------------------------------------------------

display(
    clean_vintage_analysis.head(20)
)

,vintage,months_on_books,n_loans_originated,n_active,n_defaulted_cumulative,cumulative_default_rate,marginal_default_rate,avg_pd_at_origination,avg_credit_score
0,2015Q1,1,1415,1415,3,0.002120,0.002120,0.021226,714.2
1,2015Q1,2,1415,1412,4,0.002827,0.000708,0.021226,714.2
2,2015Q1,3,1415,1411,4,0.002827,0.000000,0.021226,714.2
3,2015Q1,4,1415,1411,5,0.003534,0.000709,0.021226,714.2
4,2015Q1,5,1415,1410,10,0.007067,0.003546,0.021226,714.2
5,2015Q1,6,1415,1405,13,0.009187,0.002135,0.021226,714.2
6,2015Q1,7,1415,1402,16,0.011307,0.002140,0.021226,714.2
7,2015Q1,8,1415,1399,19,0.013428,0.002144,0.021226,714.2
8,2015Q1,9,1415,1396,21,0.014841,0.001433,0.021226,714.2
9,2015Q1,10,1415,1394,27,0.019081,0.004304,0.021226,714.2


In [94]:
# ---------------------------------------------------------------------------
# Validate vintage-month grain
# ---------------------------------------------------------------------------

vintage_grain = (
    clean_vintage_analysis
    .groupby(["vintage", "months_on_books"])
    .size()
    .reset_index(name="record_count")
)

duplicate_vintage_months = vintage_grain.loc[
    vintage_grain["record_count"] > 1
]

print(
    f"Duplicate vintage-month combinations: "
    f"{len(duplicate_vintage_months):,}"
)

display(duplicate_vintage_months)

Duplicate vintage-month combinations: 0


,vintage,months_on_books,record_count


In [95]:
# ---------------------------------------------------------------------------
# Inspect vintage and months-on-books coverage
# ---------------------------------------------------------------------------

vintage_summary = pd.Series(
    {
        "unique_vintages": clean_vintage_analysis["vintage"].nunique(),
        "minimum_months_on_books": (
            clean_vintage_analysis["months_on_books"].min()
        ),
        "maximum_months_on_books": (
            clean_vintage_analysis["months_on_books"].max()
        ),
        "missing_vintage": (
            clean_vintage_analysis["vintage"].isna().sum()
        ),
        "missing_months_on_books": (
            clean_vintage_analysis["months_on_books"].isna().sum()
        ),
    },
    name="value",
)

display(vintage_summary.to_frame())

,value
unique_vintages,36
minimum_months_on_books,1
maximum_months_on_books,60
missing_vintage,0
missing_months_on_books,0


In [96]:
# ---------------------------------------------------------------------------
# Validate cumulative default rate
# ---------------------------------------------------------------------------

clean_vintage_analysis["calculated_cumulative_default_rate"] = (
    clean_vintage_analysis["n_defaulted_cumulative"]
    / clean_vintage_analysis["n_loans_originated"]
)

clean_vintage_analysis["cumulative_default_difference"] = (
    clean_vintage_analysis["cumulative_default_rate"]
    - clean_vintage_analysis["calculated_cumulative_default_rate"]
)

display(
    clean_vintage_analysis[
        [
            "vintage",
            "months_on_books",
            "n_loans_originated",
            "n_defaulted_cumulative",
            "cumulative_default_rate",
            "calculated_cumulative_default_rate",
            "cumulative_default_difference",
        ]
    ].head(20)
)

,vintage,months_on_books,n_loans_originated,n_defaulted_cumulative,cumulative_default_rate,calculated_cumulative_default_rate,cumulative_default_difference
0,2015Q1,1,1415,3,0.002120,0.002120,-1.413428e-07
1,2015Q1,2,1415,4,0.002827,0.002827,1.448763e-07
2,2015Q1,3,1415,4,0.002827,0.002827,1.448763e-07
3,2015Q1,4,1415,5,0.003534,0.003534,4.310954e-07
4,2015Q1,5,1415,10,0.007067,0.007067,-1.378092e-07
5,2015Q1,6,1415,13,0.009187,0.009187,-2.791519e-07
6,2015Q1,7,1415,16,0.011307,0.011307,-4.204947e-07
7,2015Q1,8,1415,19,0.013428,0.013428,4.381625e-07
8,2015Q1,9,1415,21,0.014841,0.014841,1.060071e-08
9,2015Q1,10,1415,27,0.019081,0.019081,-2.720848e-07


In [97]:
# ---------------------------------------------------------------------------
# Cumulative default-rate validation
# ---------------------------------------------------------------------------

cumulative_default_validation = (
    clean_vintage_analysis["cumulative_default_difference"]
    .abs()
    .describe()
)

display(
    cumulative_default_validation.to_frame(
        "absolute_difference"
    )
)

,absolute_difference
count,2.160000e+03
mean,2.505733e-07
std,1.447212e-07
min,0.000000e+00
25%,1.228070e-07
50%,2.490242e-07
75%,3.756496e-07
max,5.000000e-07


### Vintage Cumulative Default-Rate Validation

The supplied `cumulative_default_rate` was independently reconstructed as:

**Cumulative Default Rate = Cumulative Defaults / Loans Originated**

Across all **2,160 vintage-month observations**:

- Mean absolute difference: **2.51 × 10⁻⁷**
- Median absolute difference: **2.49 × 10⁻⁷**
- Maximum absolute difference: **5.00 × 10⁻⁷**

**Conclusion:** The supplied cumulative default rate is internally consistent
with the underlying cumulative-default and origination counts, with only
negligible rounding differences.

In [98]:
# ---------------------------------------------------------------------------
# Validate marginal default rate
# ---------------------------------------------------------------------------

vintage_for_validation = (
    clean_vintage_analysis
    .sort_values(
        ["vintage", "months_on_books"]
    )
    .copy()
)

vintage_for_validation["previous_cumulative_defaults"] = (
    vintage_for_validation
    .groupby("vintage")["n_defaulted_cumulative"]
    .shift(1)
    .fillna(0)
)

vintage_for_validation["calculated_marginal_defaults"] = (
    vintage_for_validation["n_defaulted_cumulative"]
    - vintage_for_validation["previous_cumulative_defaults"]
)

vintage_for_validation["calculated_marginal_default_rate"] = (
    vintage_for_validation["calculated_marginal_defaults"]
    / vintage_for_validation["n_loans_originated"]
)

vintage_for_validation["marginal_default_rate_difference"] = (
    vintage_for_validation["marginal_default_rate"]
    - vintage_for_validation["calculated_marginal_default_rate"]
)

display(
    vintage_for_validation[
        [
            "vintage",
            "months_on_books",
            "n_defaulted_cumulative",
            "marginal_default_rate",
            "calculated_marginal_default_rate",
            "marginal_default_rate_difference",
        ]
    ].head(20)
)

,vintage,months_on_books,n_defaulted_cumulative,marginal_default_rate,calculated_marginal_default_rate,marginal_default_rate_difference
0,2015Q1,1,3,0.002120,0.002120,-1.413428e-07
1,2015Q1,2,4,0.000708,0.000707,1.286219e-06
2,2015Q1,3,4,0.000000,0.000000,0.000000e+00
3,2015Q1,4,5,0.000709,0.000707,2.286219e-06
4,2015Q1,5,10,0.003546,0.003534,1.243110e-05
5,2015Q1,6,13,0.002135,0.002120,1.485866e-05
6,2015Q1,7,16,0.002140,0.002120,1.985866e-05
7,2015Q1,8,19,0.002144,0.002120,2.385866e-05
8,2015Q1,9,21,0.001433,0.001413,1.957244e-05
9,2015Q1,10,27,0.004304,0.004240,6.371731e-05


In [99]:
# ---------------------------------------------------------------------------
# Marginal default-rate validation summary
# ---------------------------------------------------------------------------

marginal_default_validation = (
    vintage_for_validation[
        "marginal_default_rate_difference"
    ]
    .abs()
    .describe()
)

display(
    marginal_default_validation.to_frame(
        "absolute_difference"
    )
)

,absolute_difference
count,2160.000000
mean,0.000859
std,0.002852
min,0.000000
25%,0.000003
50%,0.000168
75%,0.000637
max,0.039357


In [101]:
# ---------------------------------------------------------------------------
# Test alternative marginal default-rate denominator
# ---------------------------------------------------------------------------

vintage_for_validation["calculated_marginal_rate_active"] = (
    vintage_for_validation["calculated_marginal_defaults"]
    / vintage_for_validation["n_active"]
)

vintage_for_validation["marginal_rate_active_difference"] = (
    vintage_for_validation["marginal_default_rate"]
    - vintage_for_validation["calculated_marginal_rate_active"]
)

marginal_rate_comparison = pd.DataFrame(
    {
        "original_denominator_mean_abs_diff": [
            vintage_for_validation[
                "marginal_default_rate_difference"
            ].abs().mean()
        ],
        "active_denominator_mean_abs_diff": [
            vintage_for_validation[
                "marginal_rate_active_difference"
            ].abs().mean()
        ],
        "original_denominator_max_abs_diff": [
            vintage_for_validation[
                "marginal_default_rate_difference"
            ].abs().max()
        ],
        "active_denominator_max_abs_diff": [
            vintage_for_validation[
                "marginal_rate_active_difference"
            ].abs().max()
        ],
    }
)

display(marginal_rate_comparison)

,original_denominator_mean_abs_diff,active_denominator_mean_abs_diff,original_denominator_max_abs_diff,active_denominator_max_abs_diff
0,0.000859,1.909064e-07,0.039357,5.000000e-07


In [102]:
# ---------------------------------------------------------------------------
# Compare marginal-rate definitions
# ---------------------------------------------------------------------------

display(
    vintage_for_validation[
        [
            "vintage",
            "months_on_books",
            "n_defaulted_cumulative",
            "n_active",
            "calculated_marginal_defaults",
            "marginal_default_rate",
            "calculated_marginal_default_rate",
            "calculated_marginal_rate_active",
        ]
    ].head(20)
)

,vintage,months_on_books,n_defaulted_cumulative,n_active,calculated_marginal_defaults,marginal_default_rate,calculated_marginal_default_rate,calculated_marginal_rate_active
0,2015Q1,1,3,1415,3.0,0.002120,0.002120,0.002120
1,2015Q1,2,4,1412,1.0,0.000708,0.000707,0.000708
2,2015Q1,3,4,1411,0.0,0.000000,0.000000,0.000000
3,2015Q1,4,5,1411,1.0,0.000709,0.000707,0.000709
4,2015Q1,5,10,1410,5.0,0.003546,0.003534,0.003546
5,2015Q1,6,13,1405,3.0,0.002135,0.002120,0.002135
6,2015Q1,7,16,1402,3.0,0.002140,0.002120,0.002140
7,2015Q1,8,19,1399,3.0,0.002144,0.002120,0.002144
8,2015Q1,9,21,1396,2.0,0.001433,0.001413,0.001433
9,2015Q1,10,27,1394,6.0,0.004304,0.004240,0.004304


### Marginal Default Rate Validation

The supplied `marginal_default_rate` is not calculated using the original
number of loans originated.

It is consistent with:

**Marginal Default Rate = New Defaults / Active Loans**

where:

**New Defaults = Current Cumulative Defaults − Previous Cumulative Defaults**

Across all 2,160 vintage-month observations:

- Mean absolute difference: **1.91 × 10⁻⁷**
- Maximum absolute difference: **5.00 × 10⁻⁷**

**Conclusion:** The supplied marginal default rate is internally consistent
with an active-loan denominator, subject only to negligible rounding.

In [103]:
# ---------------------------------------------------------------------------
# Validate active-loan counts
# ---------------------------------------------------------------------------

active_loan_checks = pd.Series(
    {
        "active_above_originated": (
            clean_vintage_analysis["n_active"]
            > clean_vintage_analysis["n_loans_originated"]
        ).sum(),
        "negative_active_loans": (
            clean_vintage_analysis["n_active"] < 0
        ).sum(),
        "negative_cumulative_defaults": (
            clean_vintage_analysis["n_defaulted_cumulative"] < 0
        ).sum(),
        "cumulative_defaults_above_originated": (
            clean_vintage_analysis["n_defaulted_cumulative"]
            > clean_vintage_analysis["n_loans_originated"]
        ).sum(),
    },
    name="count",
)

display(active_loan_checks.to_frame())

,count
active_above_originated,0
negative_active_loans,0
negative_cumulative_defaults,0
cumulative_defaults_above_originated,0


In [104]:
# ---------------------------------------------------------------------------
# Validate cumulative defaults are non-decreasing within each vintage
# ---------------------------------------------------------------------------

vintage_sequence = (
    clean_vintage_analysis
    .sort_values(
        ["vintage", "months_on_books"]
    )
    .copy()
)

vintage_sequence["cumulative_default_change"] = (
    vintage_sequence
    .groupby("vintage")["n_defaulted_cumulative"]
    .diff()
)

negative_cumulative_changes = vintage_sequence.loc[
    vintage_sequence["cumulative_default_change"] < 0
]

print(
    f"Vintages with decreasing cumulative defaults: "
    f"{len(negative_cumulative_changes):,}"
)

display(
    negative_cumulative_changes[
        [
            "vintage",
            "months_on_books",
            "n_defaulted_cumulative",
            "cumulative_default_change",
        ]
    ].head(20)
)

Vintages with decreasing cumulative defaults: 0


,vintage,months_on_books,n_defaulted_cumulative,cumulative_default_change


In [105]:
# ---------------------------------------------------------------------------
# Validate active-loan and cumulative-default relationships
# ---------------------------------------------------------------------------

vintage_quality_checks = pd.Series(
    {
        "active_above_originated": (
            clean_vintage_analysis["n_active"]
            > clean_vintage_analysis["n_loans_originated"]
        ).sum(),
        "negative_active_loans": (
            clean_vintage_analysis["n_active"] < 0
        ).sum(),
        "negative_cumulative_defaults": (
            clean_vintage_analysis["n_defaulted_cumulative"] < 0
        ).sum(),
        "cumulative_defaults_above_originated": (
            clean_vintage_analysis["n_defaulted_cumulative"]
            > clean_vintage_analysis["n_loans_originated"]
        ).sum(),
    },
    name="count",
)

display(vintage_quality_checks.to_frame())

,count
active_above_originated,0
negative_active_loans,0
negative_cumulative_defaults,0
cumulative_defaults_above_originated,0


In [106]:
# ---------------------------------------------------------------------------
# Validate cumulative defaults are non-decreasing
# ---------------------------------------------------------------------------

vintage_sequence = (
    clean_vintage_analysis
    .sort_values(
        ["vintage", "months_on_books"]
    )
    .copy()
)

vintage_sequence["cumulative_default_change"] = (
    vintage_sequence
    .groupby("vintage")["n_defaulted_cumulative"]
    .diff()
)

negative_cumulative_changes = vintage_sequence.loc[
    vintage_sequence["cumulative_default_change"] < 0
]

print(
    "Records with decreasing cumulative defaults: "
    f"{len(negative_cumulative_changes):,}"
)

display(
    negative_cumulative_changes[
        [
            "vintage",
            "months_on_books",
            "n_defaulted_cumulative",
            "cumulative_default_change",
        ]
    ].head(20)
)

Records with decreasing cumulative defaults: 0


,vintage,months_on_books,n_defaulted_cumulative,cumulative_default_change


In [107]:
# ---------------------------------------------------------------------------
# Validate months-on-books sequence within each vintage
# ---------------------------------------------------------------------------

vintage_sequence["mob_gap"] = (
    vintage_sequence
    .groupby("vintage")["months_on_books"]
    .diff()
)

mob_gap_summary = (
    vintage_sequence.loc[
        vintage_sequence["mob_gap"].notna(),
        "mob_gap",
    ]
    .value_counts()
    .sort_index()
    .rename("count")
    .to_frame()
)

display(mob_gap_summary)

,count
mob_gap,
1.0,2124


In [108]:
# ---------------------------------------------------------------------------
# Validate origination-level risk metrics
# ---------------------------------------------------------------------------

origination_metric_checks = pd.Series(
    {
        "avg_pd_below_zero": (
            clean_vintage_analysis["avg_pd_at_origination"] < 0
        ).sum(),
        "avg_pd_above_one": (
            clean_vintage_analysis["avg_pd_at_origination"] > 1
        ).sum(),
        "avg_credit_score_below_zero": (
            clean_vintage_analysis["avg_credit_score"] < 0
        ).sum(),
        "avg_credit_score_above_1000": (
            clean_vintage_analysis["avg_credit_score"] > 1000
        ).sum(),
    },
    name="count",
)

display(origination_metric_checks.to_frame())

,count
avg_pd_below_zero,0
avg_pd_above_one,0
avg_credit_score_below_zero,0
avg_credit_score_above_1000,0


### Vintage Analysis Validation — Finding

The vintage dataset passes the key structural and range validations.

- Cumulative defaults decrease within a vintage: **0 records**
- Non-consecutive months-on-books transitions: **0 records**
- Invalid average PD values: **0 records**
- Invalid average credit scores: **0 records**

All observed vintage-month sequences advance by exactly one month.

**Conclusion:** The vintage analysis table is structurally consistent and
requires no corrective treatment for the validated fields.

In [109]:
# ---------------------------------------------------------------------------
# Remove temporary vintage validation columns
# ---------------------------------------------------------------------------

TEMPORARY_VINTAGE_COLUMNS = [
    "calculated_cumulative_default_rate",
    "cumulative_default_difference",
    "previous_cumulative_defaults",
    "calculated_marginal_defaults",
    "calculated_marginal_default_rate",
    "marginal_default_rate_difference",
    "calculated_marginal_rate_active",
    "marginal_rate_active_difference",
    "mob_gap",
    "cumulative_default_change",
]

clean_vintage_analysis = clean_vintage_analysis.drop(
    columns=[
        column
        for column in TEMPORARY_VINTAGE_COLUMNS
        if column in clean_vintage_analysis.columns
    ]
)

print("Temporary vintage validation columns removed.")

Temporary vintage validation columns removed.


In [110]:
# ---------------------------------------------------------------------------
# Save cleaned vintage analysis
# ---------------------------------------------------------------------------

CLEAN_VINTAGE_FILE = (
    PROCESSED_DATA_DIR / "vintage_analysis_clean.csv"
)

clean_vintage_analysis.to_csv(
    CLEAN_VINTAGE_FILE,
    index=False,
)

print(
    f"Saved cleaned vintage analysis to:\n"
    f"{CLEAN_VINTAGE_FILE}"
)

Saved cleaned vintage analysis to:
D:\Bank_sense_2.0\data\processed\vintage_analysis_clean.csv


In [111]:
# ---------------------------------------------------------------------------
# Verify processed output
# ---------------------------------------------------------------------------

if not CLEAN_VINTAGE_FILE.exists():
    raise FileNotFoundError(
        f"Processed file was not created: {CLEAN_VINTAGE_FILE}"
    )

print(f"Verified: {CLEAN_VINTAGE_FILE.name}")

Verified: vintage_analysis_clean.csv


In [112]:
# ---------------------------------------------------------------------------
# Cross-dataset sector consistency
# ---------------------------------------------------------------------------

sector_sets = {
    "loan_portfolio": set(clean_loan_portfolio["sector"].dropna().unique()),
    "credit_ratings": set(clean_credit_ratings["sector"].dropna().unique()),
    "macro_stress": set(clean_macro_stress["sector"].dropna().unique()),
}

sector_comparison = {
    name: sorted(values)
    for name, values in sector_sets.items()
}

display(sector_comparison)

common_sectors = set.intersection(*sector_sets.values())
all_sectors = set.union(*sector_sets.values())

print(f"Common sectors: {len(common_sectors)}")
print(f"Total unique sectors: {len(all_sectors)}")
print(f"Sector mismatch: {all_sectors - common_sectors}")

{'loan_portfolio': ['Consumer',
  'Energy',
  'Financials',
  'Healthcare',
  'Industrials',
  'Real_Estate',
  'Retail',
  'Technology',
  'Telecom',
  'Utilities'],
 'credit_ratings': ['Consumer',
  'Energy',
  'Financials',
  'Healthcare',
  'Industrials',
  'Real_Estate',
  'Retail',
  'Technology',
  'Telecom',
  'Utilities'],
 'macro_stress': ['Consumer',
  'Energy',
  'Financials',
  'Healthcare',
  'Industrials',
  'Real_Estate',
  'Retail',
  'Technology',
  'Telecom',
  'Utilities']}

Common sectors: 10
Total unique sectors: 10
Sector mismatch: set()


In [113]:
# ---------------------------------------------------------------------------
# Cross-dataset rating consistency
# ---------------------------------------------------------------------------

loan_ratings = set(
    clean_loan_portfolio["initial_rating"]
    .dropna()
    .unique()
)

from_ratings = set(
    clean_credit_ratings["from_rating"]
    .dropna()
    .unique()
)

to_ratings = set(
    clean_credit_ratings["to_rating"]
    .dropna()
    .unique()
)

print("Loan portfolio ratings:", sorted(loan_ratings))
print("Rating-transition from ratings:", sorted(from_ratings))
print("Rating-transition to ratings:", sorted(to_ratings))

print(
    "\nRatings in loan portfolio missing from rating table:",
    loan_ratings - from_ratings,
)

Loan portfolio ratings: ['A', 'AA', 'AAA', 'B', 'BB', 'BBB', 'CCC']
Rating-transition from ratings: ['A', 'AA', 'AAA', 'B', 'BB', 'BBB', 'CCC']
Rating-transition to ratings: ['A', 'AA', 'AAA', 'B', 'BB', 'BBB', 'CCC', 'D']

Ratings in loan portfolio missing from rating table: set()


In [114]:
# ---------------------------------------------------------------------------
# Cross-dataset temporal coverage
# ---------------------------------------------------------------------------

coverage = pd.Series(
    {
        "loan_origination_start": clean_loan_portfolio["origination_date"].min(),
        "loan_origination_end": clean_loan_portfolio["origination_date"].max(),
        "loan_maturity_end": clean_loan_portfolio["maturity_date"].max(),
        "portfolio_metrics_start": clean_portfolio_metrics["date"].min(),
        "portfolio_metrics_end": clean_portfolio_metrics["date"].max(),
        "rating_year_start": clean_credit_ratings["year"].min(),
        "rating_year_end": clean_credit_ratings["year"].max(),
    },
    name="value",
)

display(coverage.to_frame())

,value
loan_origination_start,2015-01-01 00:00:00
loan_origination_end,2023-12-01 00:00:00
loan_maturity_end,2024-12-31 00:00:00
portfolio_metrics_start,2015-01-01 00:00:00
portfolio_metrics_end,2024-12-01 00:00:00
rating_year_start,2015
rating_year_end,2024


In [115]:
# ---------------------------------------------------------------------------
# Processed dataset inventory
# ---------------------------------------------------------------------------

processed_files = {
    "loan_portfolio": PROCESSED_DATA_DIR / "loan_portfolio_clean.csv",
    "credit_ratings": PROCESSED_DATA_DIR / "credit_ratings_clean.csv",
    "macro_stress": PROCESSED_DATA_DIR / "macro_stress_scenarios_clean.csv",
    "portfolio_metrics": PROCESSED_DATA_DIR / "portfolio_metrics_clean.csv",
    "vintage_analysis": PROCESSED_DATA_DIR / "vintage_analysis_clean.csv",
}

processed_inventory = pd.DataFrame(
    [
        {
            "dataset": name,
            "file": path.name,
            "exists": path.exists(),
            "size_mb": (
                round(path.stat().st_size / (1024 ** 2), 2)
                if path.exists()
                else None
            ),
        }
        for name, path in processed_files.items()
    ]
)

display(processed_inventory)

,dataset,file,exists,size_mb
0,loan_portfolio,loan_portfolio_clean.csv,True,8.83
1,credit_ratings,credit_ratings_clean.csv,True,0.90
2,macro_stress,macro_stress_scenarios_clean.csv,True,0.01
3,portfolio_metrics,portfolio_metrics_clean.csv,True,0.02
4,vintage_analysis,vintage_analysis_clean.csv,True,0.11


# Data Cleaning — Final Conclusion

All five source datasets have been independently audited, standardized, and
saved as processed analytical files.

The cleaning process preserved the original raw data and did not silently
remove unusual observations.

Key principles applied:

- Raw source files were never overwritten.
- Date and numeric fields were standardized.
- Categorical values were normalized.
- Structurally valid missing values were preserved.
- Temporal anomalies were flagged rather than silently corrected.
- Financial relationships were independently validated.
- Source-specific field definitions were investigated before transformation.
- Dataset-specific calculation fields were not overwritten when their original
  construction could not be established.

The processed datasets are now ready for cross-dataset modelling analysis.

**Next phase:** leakage analysis and modelling-variable eligibility.